# Regulatory Compliance Intelligence Multi-Agent System
## Week 8 Assignment: Production Deployment

**Course**: Agentic AI Bootcamp - Staff-Level System Design

---

## Learning Objectives

- Build the integrated compliance system
- Create Gradio interface
- Deploy as FastAPI application

## Difficulty: Advanced

**IMPORTANT**: Run all prior week cells first before starting this week's exercises.

---

---
# WEEK 1: Environment Setup & Langfuse Initialization (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 1: Environment Setup & Langfuse Initialization

---

## Learning Objectives

By the end of Week 1, you will be able to:

- [ ] Configure a production-ready development environment
- [ ] Initialize Langfuse observability from the very first line of code
- [ ] Understand the regulatory compliance document landscape
- [ ] Create traced wrapper functions for all AI operations
- [ ] Discover and catalog compliance documents automatically

## 1.1 Package Installation

We install packages in a specific order to avoid dependency conflicts. Each package serves a distinct purpose in our compliance architecture.

In [ ]:
# ============================================================================
# WEEK 1.1: PACKAGE INSTALLATION
# ============================================================================
# Pinned versions for reproducibility. Every package is verified after install.

import subprocess, sys

_packages = ["openai==1.59.6", "langfuse==2.57.1", "langchain==0.3.14", "langchain-openai==0.2.14", "langchain-community==0.3.14", "langchain-core==0.3.29", "chromadb", "networkx", "pyvis", "python-docx", "pdfplumber", "PyPDF2", "plotly", "seaborn", "pydantic>=2.0", "tenacity", "rich", "gradio", "python-dotenv", "numpy", "pandas", "matplotlib"]

print("Installing packages (this may take 1-2 minutes)...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q"] + _packages,
    capture_output=True, text=True
)

if result.returncode != 0:
    for line in result.stderr.split("\n"):
        if line.strip() and "dependency resolver" not in line.lower() and "notice" not in line.lower():
            print(line)

# ---------- Verify EVERY import ----------
_verify = [
    ("openai", "openai"),
    ("langfuse", "langfuse"),
    ("langchain", "langchain"),
    ("langchain_openai", "langchain_openai"),
    ("chromadb", "chromadb"),
    ("networkx", "networkx"),
    ("pyvis", "pyvis.network"),
    ("docx", "docx"),
    ("pdfplumber", "pdfplumber"),
    ("plotly", "plotly"),
    ("seaborn", "seaborn"),
    ("pydantic", "pydantic"),
    ("tenacity", "tenacity"),
    ("rich", "rich"),
    ("gradio", "gradio"),
    ("dotenv", "dotenv"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
]

_failed = []
for name, imp in _verify:
    try:
        __import__(imp)
    except ImportError:
        _failed.append(name)

if _failed:
    msg = f"FATAL: These packages failed to import: {', '.join(_failed)}\n"
    msg += "Try: Runtime > Restart runtime, then re-run this cell."
    raise ImportError(msg)

print("=" * 60)
print(f"All {len(_verify)} packages installed and verified!")
print("=" * 60)

## Key Concept: Package Architecture for Compliance Systems

Understanding why we use each package helps you make informed architectural decisions:

| Package | Purpose | Why It Matters for Compliance |
|---------|---------|---------|
| `langchain` | Orchestration framework | Chains multiple AI operations into auditable workflows |
| `langfuse` | Observability platform | **Critical**: Audit trail for all AI decisions |
| `chromadb` | Vector database | Semantic search across regulations and policies |
| `pdfplumber` | PDF extraction | Handles complex regulatory PDF formats |
| `python-docx` | DOCX processing | Parses company policy documents |
| `networkx` | Graph analysis | Maps regulation-to-policy relationships |
| `pydantic` | Data validation | Ensures structured, validated compliance outputs |

### Why Langfuse is Non-Negotiable for Compliance

In regulatory compliance, **every AI decision must be explainable and auditable**. When a regulator asks:
- "Why did your system flag this policy as non-compliant?"
- "What data informed this compliance score?"
- "Show me the audit trail for this gap assessment."

Langfuse provides complete visibility into every LLM call, embedding generation, and agent decision.

## 1.2 Environment Configuration

This section handles the critical task of detecting our runtime environment and loading API keys securely.

In [ ]:
# ============================================================================
# WEEK 1.2: ENVIRONMENT DETECTION
# ============================================================================

import os
import sys
import json
import re
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from typing import List, Dict, Optional, Any, Tuple
from dataclasses import dataclass, field
from enum import Enum
import warnings
warnings.filterwarnings('ignore')

# Detect runtime environment
IN_COLAB = 'google.colab' in sys.modules

print(f"Environment: {'Google Colab' if IN_COLAB else 'Local/Jupyter'}")
print(f"Python version: {sys.version.split()[0]}")

In [ ]:
# ============================================================================
# API KEY CONFIGURATION & DATA INGESTION
# ============================================================================
# API keys from Colab Secrets or .env file
# Datasets fetched programmatically from GitHub

import subprocess

DATASET_REPO = "https://github.com/AI-Project-Lab/IK-pwc-agenticai-datasets.git"
DATASET_PROJECT = "regulatory_compliance"

# Project identifiers
PROJECT_NAME = "regulatory-compliance-system"
PROJECT_VERSION = "1.0.0"

if IN_COLAB:
    print("Configuring for Google Colab environment...")

    # --- API Key Configuration ---
    try:
        from google.colab import userdata
        os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
        os.environ['LANGFUSE_SECRET_KEY'] = userdata.get('LANGFUSE_SECRET_KEY')
        os.environ['LANGFUSE_PUBLIC_KEY'] = userdata.get('LANGFUSE_PUBLIC_KEY')
        _host = userdata.get('LANGFUSE_HOST') or ''
        os.environ['LANGFUSE_HOST'] = _host if _host.startswith('http') else 'https://cloud.langfuse.com'
        print("API keys loaded from Colab Secrets")
    except Exception as e:
        print(f"Colab Secrets not available: {e}")
        import getpass
        os.environ['OPENAI_API_KEY'] = getpass.getpass('Enter OpenAI API Key: ')
        os.environ['LANGFUSE_SECRET_KEY'] = getpass.getpass('Enter Langfuse Secret Key: ')
        os.environ['LANGFUSE_PUBLIC_KEY'] = getpass.getpass('Enter Langfuse Public Key: ')
        os.environ['LANGFUSE_HOST'] = 'https://cloud.langfuse.com'

    # --- Dataset Ingestion from GitHub ---
    dataset_path = "/content/datasets"
    if not os.path.exists(f"{dataset_path}/{DATASET_PROJECT}"):
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, dataset_path],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available.")
    DATA_DIR_BASE = f"{dataset_path}/{DATASET_PROJECT}"
else:
    print("Configuring for local environment...")
    from dotenv import load_dotenv
    load_dotenv()

    # --- Dataset Ingestion from GitHub ---
    datasets_parent = Path('.').resolve().parent.parent / 'datasets'
    if not (datasets_parent / DATASET_PROJECT).exists():
        print(f"\nCloning dataset from {DATASET_REPO}...")
        subprocess.run(["git", "clone", "--depth", "1", DATASET_REPO, str(datasets_parent)],
                       check=True, capture_output=True)
        print("Dataset cloned successfully!")
    else:
        print("\nDataset already available locally.")
    DATA_DIR_BASE = str(datasets_parent / DATASET_PROJECT)

REGULATIONS_DIR_BASE = f'{DATA_DIR_BASE}/Regulations'
POLICIES_DIR_BASE = f'{DATA_DIR_BASE}/Company Policies sample documents'

# Convert to Path objects for easier manipulation
REGULATIONS_DIR = Path(REGULATIONS_DIR_BASE)
POLICIES_DIR = Path(POLICIES_DIR_BASE)

# Validate API keys
required_keys = ['OPENAI_API_KEY', 'LANGFUSE_SECRET_KEY', 'LANGFUSE_PUBLIC_KEY']
missing_keys = [key for key in required_keys if not os.environ.get(key)]
if missing_keys:
    raise EnvironmentError(f"Missing required API keys: {missing_keys}")

DATA_DIR = Path(DATA_DIR_BASE)
print(f"\nAll API keys validated successfully")
print(f"Data directory: {DATA_DIR}")
print(f"Directory exists: {DATA_DIR.exists()}")
if DATA_DIR.exists():
    file_count = sum(1 for _ in DATA_DIR.rglob('*') if _.is_file())
    print(f"Total files found: {file_count}")

## Deep Dive: Secure API Key Management for Compliance Systems

API keys are the "passwords" to your AI services. In a compliance context, improper key handling can lead to:
- **Data breaches** if keys are exposed in code repositories
- **Compliance violations** under data protection regulations (GDPR, etc.)
- **Audit failures** if key management isn't documented

### Best Practices:

| Environment | Method | Security Level |
|-------------|--------|---------------|
| Google Colab | Colab Secrets | High |
| Local Development | `.env` files (gitignored) | Medium |
| Production | Cloud secrets manager (AWS Secrets, GCP Secret Manager) | Highest |

### Never Do This:
```python
# WRONG - Never hardcode keys!
OPENAI_API_KEY = "sk-abc123..."

# WRONG - Never commit .env files!
# Make sure .env is in .gitignore
```

### Compliance Requirement:
Most regulatory frameworks (SOC 2, ISO 27001, IRDAI cybersecurity guidelines) require:
- Secrets to be stored separately from code
- Key rotation policies (90 days recommended)
- Audit logging of key usage

## 1.3 Langfuse Initialization

Now we initialize Langfuse - this is the **foundation of our observability strategy**. Every subsequent operation will be traceable and auditable.

In [ ]:
# ============================================================================
# WEEK 1.3: LANGFUSE INITIALIZATION
# ============================================================================

from langfuse import Langfuse
try:
    from langfuse.callback import CallbackHandler as LangfuseCallbackHandler
except (ImportError, ModuleNotFoundError):
    try:
        from langfuse.langchain import CallbackHandler as LangfuseCallbackHandler
    except (ImportError, ModuleNotFoundError):
        # Langfuse v3+: CallbackHandler not needed, create a no-op
        class LangfuseCallbackHandler:
            def __init__(self, **kwargs): pass
from openai import OpenAI

# Initialize Langfuse client
langfuse = Langfuse(
    secret_key=os.environ.get('LANGFUSE_SECRET_KEY'),
    public_key=os.environ.get('LANGFUSE_PUBLIC_KEY'),
    host=os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')
)

# Verify connection
langfuse.auth_check()

# Create unique session ID for this notebook run
# This groups all traces together for easy filtering
SESSION_ID = f"compliance-master-{datetime.now().strftime('%Y%m%d-%H%M%S')}"

# Initialize OpenAI client
openai_client = OpenAI()

print(f"Langfuse connected successfully!")
print(f"Session ID: {SESSION_ID}")
print(f"Dashboard: {os.environ.get('LANGFUSE_HOST', 'https://cloud.langfuse.com')}")

## Key Concept: Session IDs and Trace Organization

In Langfuse, traces are organized hierarchically. Understanding this hierarchy is crucial for compliance auditing:

```
Project: regulatory-compliance-system
  └── Session: compliance-master-20240115-143022
       ├── Trace: document-discovery
       │    └── Span: scan-regulations
       │    └── Span: scan-policies
       ├── Trace: requirement-extraction
       │    └── Generation: extract-requirements-REG-001
       │    └── Generation: extract-requirements-REG-002
       ├── Trace: gap-detection-governance
       │    └── Span: find-relevant-regulations
       │    └── Span: find-matching-policies
       │    └── Generation: assess-gap
       │    └── Score: gap-severity (0.75)
       └── Trace: compliance-score-final
            └── Score: overall-compliance (0.72)
```

### Trace Types:
- **Trace**: Top-level operation (e.g., "assess governance compliance")
- **Span**: Sub-operation within a trace (e.g., "search vector store")
- **Generation**: LLM call with input/output (e.g., "gap analysis prompt")
- **Score**: Evaluation metric (e.g., "compliance score: 72%")

## 1.4 Traced Wrapper Functions

We create wrapper functions that automatically trace all OpenAI API calls. This ensures **100% observability** without cluttering our application code.

In [ ]:
# ============================================================================
# WEEK 1.4: TRACED EMBEDDING FUNCTION
# ============================================================================

def traced_embedding(
    text: str,
    trace_name: str = "embedding",
    metadata: Dict = None,
    model: str = "text-embedding-3-small"
) -> List[float]:
    """
    Generate embedding with full Langfuse tracing.

    Every embedding operation is visible in Langfuse, including:
    - Input text (truncated for display)
    - Model used
    - Token usage
    - Output dimensions

    Args:
        text: Text to embed
        trace_name: Name for the trace (appears in Langfuse)
        metadata: Additional metadata to attach
        model: OpenAI embedding model to use

    Returns:
        List of floats representing the embedding vector
    """
    # Create trace for this embedding operation
    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        tags=["embedding", "compliance"],
        metadata=metadata or {}
    )

    # Create generation span for the API call
    generation = trace.generation(
        name="openai-embedding",
        model=model,
        input=text[:500] + "..." if len(text) > 500 else text
    )

    # Make API call
    response = openai_client.embeddings.create(
        model=model,
        input=text[:8000]  # Truncate to model limit
    )
    embedding = response.data[0].embedding

    # End generation with output and usage
    generation.end(
        output={"dimensions": len(embedding)},
        usage={
            "input": response.usage.prompt_tokens,
            "total": response.usage.total_tokens
        }
    )

    return embedding

print("traced_embedding() function defined")

In [ ]:
# ============================================================================
# TRACED COMPLETION FUNCTION
# ============================================================================

def traced_completion(
    prompt: str,
    trace_name: str = "completion",
    model: str = "gpt-4o-mini",
    temperature: float = 0,
    metadata: Dict = None,
    system_prompt: str = None
) -> str:
    """
    Generate completion with full Langfuse tracing.

    Traces include:
    - Full prompt (truncated for display)
    - Model and temperature
    - Token usage (input, output, total)
    - Full response

    Args:
        prompt: User prompt
        trace_name: Name for the trace
        model: OpenAI model to use
        temperature: Sampling temperature (0 for deterministic)
        metadata: Additional metadata
        system_prompt: Optional system prompt

    Returns:
        Model response as string
    """
    trace = langfuse.trace(
        name=trace_name,
        session_id=SESSION_ID,
        tags=["completion", "compliance"],
        metadata=metadata or {}
    )

    # Build messages
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.append({"role": "user", "content": prompt})

    generation = trace.generation(
        name="openai-completion",
        model=model,
        input=prompt[:1000] if len(prompt) > 1000 else prompt
    )

    response = openai_client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature
    )
    result = response.choices[0].message.content

    generation.end(
        output=result[:500] if len(result) > 500 else result,
        usage={
            "input": response.usage.prompt_tokens,
            "output": response.usage.completion_tokens,
            "total": response.usage.total_tokens
        }
    )

    return result

print("traced_completion() function defined")

In [ ]:
# ============================================================================
# LANGCHAIN CALLBACK HANDLER FACTORY
# ============================================================================

def get_langfuse_handler(
    trace_name: str,
    tags: List[str] = None,
    metadata: Dict = None
) -> LangfuseCallbackHandler:
    """
    Get LangChain callback handler for Langfuse tracing.

    Use this with all LangChain chains and agents to ensure
    complete observability.

    Example:
        handler = get_langfuse_handler("gap-detection", ["agent", "gap"])
        result = chain.invoke(input, config={"callbacks": [handler]})

    Args:
        trace_name: Name for the trace
        tags: List of tags for filtering in Langfuse
        metadata: Additional metadata

    Returns:
        Configured LangfuseCallbackHandler
    """
    return LangfuseCallbackHandler(
        secret_key=os.environ.get('LANGFUSE_SECRET_KEY'),
        public_key=os.environ.get('LANGFUSE_PUBLIC_KEY'),
        host=os.environ.get('LANGFUSE_HOST'),
        session_id=SESSION_ID,
        trace_name=trace_name,
        tags=tags or ["compliance"],
        metadata=metadata or {}
    )

print("get_langfuse_handler() function defined")
print("\nTraced helper functions ready:")
print("  - traced_embedding(): Generate embeddings with full tracing")
print("  - traced_completion(): Generate completions with full tracing")
print("  - get_langfuse_handler(): Get LangChain callback for any chain/agent")

## Example: Testing Our Traced Functions

Let's verify our traced functions work correctly and observe the traces in Langfuse.

In [ ]:
# ============================================================================
# EXAMPLE 1: Test Embedding Function
# ============================================================================

print("EXAMPLE 1: Testing traced_embedding()")
print("=" * 50)

test_text = "Insurance companies must comply with IRDAI governance regulations."

embedding = traced_embedding(
    text=test_text,
    trace_name="test-embedding-compliance",
    metadata={"test": True, "example": 1}
)

print(f"Input: '{test_text}'")
print(f"Output: Vector with {len(embedding)} dimensions")
print(f"First 5 values: {embedding[:5]}")
print(f"\nCheck Langfuse for trace: 'test-embedding-compliance'")

In [ ]:
# ============================================================================
# EXAMPLE 2: Test Completion Function
# ============================================================================

print("EXAMPLE 2: Testing traced_completion()")
print("=" * 50)

prompt = "What are the key components of an effective insurance compliance program? List 3 points briefly."

response = traced_completion(
    prompt=prompt,
    trace_name="test-completion-compliance",
    metadata={"test": True, "example": 2}
)

print(f"Prompt: '{prompt[:80]}...'")
print(f"\nResponse:\n{response}")
print(f"\nCheck Langfuse for trace: 'test-completion-compliance'")

In [ ]:
# ============================================================================
# EXAMPLE 3: Batch Embeddings (Common Pattern in Compliance)
# ============================================================================

print("EXAMPLE 3: Batch Embeddings for Compliance Concepts")
print("=" * 50)

compliance_concepts = [
    "Board of directors governance requirements",
    "Risk management framework implementation",
    "Cybersecurity controls and data protection",
    "Internal audit procedures and controls",
    "Fraud prevention and detection mechanisms"
]

embeddings_batch = []
for i, concept in enumerate(compliance_concepts, 1):
    emb = traced_embedding(
        text=concept,
        trace_name=f"batch-embedding-{i}",
        metadata={"batch": "compliance-concepts", "index": i}
    )
    embeddings_batch.append(emb)
    print(f"  {i}. '{concept[:40]}...' -> {len(emb)} dims")

print(f"\nGenerated {len(embeddings_batch)} embeddings")
print(f"Check Langfuse for traces: 'batch-embedding-*'")

# Flush traces to Langfuse
langfuse.flush()

## 1.5 Regulatory Compliance Taxonomy

Before processing documents, we need to understand the regulatory landscape. This taxonomy maps compliance domains to their regulations and policies.

In [ ]:
# ============================================================================
# WEEK 1.5: COMPLIANCE DOMAIN TAXONOMY
# ============================================================================

COMPLIANCE_DOMAINS = {
    'governance': {
        'description': 'Corporate governance and board structure',
        'regulator': 'IRDAI',
        'regulations': ['corporate-governance-guidelines'],
        'policies': ['Governance and Corporate Structure'],
        'risk_weight': 0.95,
        'audit_frequency': 'Annual'
    },
    'risk_management': {
        'description': 'Enterprise risk management frameworks',
        'regulator': 'MAS/IRDAI',
        'regulations': ['MAS Risk management Guidelines', 'fraud-risk-framework'],
        'policies': ['Risk Management and Compliance', 'Underwriting and Risk Assessment'],
        'risk_weight': 0.95,
        'audit_frequency': 'Quarterly'
    },
    'information_security': {
        'description': 'Information security and cyber resilience',
        'regulator': 'IRDAI/MAS',
        'regulations': ['Information security Guidelines'],
        'policies': ['Cyber Insurance Policy Document'],
        'risk_weight': 0.90,
        'audit_frequency': 'Quarterly'
    },
    'claims_management': {
        'description': 'Claims processing and grievance handling',
        'regulator': 'IRDAI',
        'regulations': ['grievance-redressal-policy', 'Intro_Reinsurance_Claims'],
        'policies': [],
        'risk_weight': 0.85,
        'audit_frequency': 'Monthly'
    },
    'reinsurance': {
        'description': 'Reinsurance operations and agreements',
        'regulator': 'IRDAI',
        'regulations': ['corporate-governance-guidelines-for-reinsurance'],
        'policies': ['Quota Share Reinsurance', 'Surplus Share Reinsurance', 'Non Proportional Reinsurance'],
        'risk_weight': 0.80,
        'audit_frequency': 'Semi-Annual'
    },
    'audit': {
        'description': 'Internal audit and controls',
        'regulator': 'IRDAI',
        'regulations': ['Internal Audit of GICs', 'Audit COntol for Insurance'],
        'policies': [],
        'risk_weight': 0.90,
        'audit_frequency': 'Annual'
    },
    'sales_distribution': {
        'description': 'Sales practices and distribution channels',
        'regulator': 'MAS',
        'regulations': ['MAS Standards of Conduct', 'Guide for Insurance Sales'],
        'policies': [],
        'risk_weight': 0.75,
        'audit_frequency': 'Quarterly'
    },
    'investments': {
        'description': 'Investment management and financial controls',
        'regulator': 'IRDAI',
        'regulations': ['Insurer Investments and Financial Management'],
        'policies': [],
        'risk_weight': 0.85,
        'audit_frequency': 'Monthly'
    },
    'fraud': {
        'description': 'Fraud prevention and detection',
        'regulator': 'IRDAI',
        'regulations': ['fraud-risk-framework', 'Fraud Prevention'],
        'policies': [],
        'risk_weight': 0.90,
        'audit_frequency': 'Continuous'
    }
}

print("COMPLIANCE DOMAIN TAXONOMY")
print("=" * 70)

In [ ]:
# ============================================================================
# DISPLAY TAXONOMY WITH TRACING
# ============================================================================

# Log taxonomy to Langfuse for audit trail
taxonomy_trace = langfuse.trace(
    name="compliance-taxonomy-definition",
    session_id=SESSION_ID,
    tags=["setup", "taxonomy", "compliance"]
)

# Display domains
for domain, info in COMPLIANCE_DOMAINS.items():
    print(f"\n{domain.upper()}")
    print(f"  Description: {info['description']}")
    print(f"  Regulator: {info['regulator']}")
    print(f"  Risk Weight: {info['risk_weight']}")
    print(f"  Audit Frequency: {info['audit_frequency']}")
    print(f"  Regulations: {len(info['regulations'])}")
    print(f"  Policies: {len(info['policies'])}")

# Update trace with taxonomy summary
taxonomy_trace.update(output={
    "domains_defined": len(COMPLIANCE_DOMAINS),
    "domain_names": list(COMPLIANCE_DOMAINS.keys()),
    "total_regulations": sum(len(d['regulations']) for d in COMPLIANCE_DOMAINS.values()),
    "total_policies": sum(len(d['policies']) for d in COMPLIANCE_DOMAINS.values())
})

langfuse.flush()

## Deep Dive: Understanding IRDAI and MAS Regulations

### IRDAI (Insurance Regulatory and Development Authority of India)

IRDAI is the apex regulatory body for insurance in India, established under the IRDAI Act, 1999. Key regulatory areas:

| Area | Key Requirements | Penalty for Non-Compliance |
|------|-----------------|---------------------------|
| **Corporate Governance** | Independent directors, board committees, risk management | Up to INR 25 lakhs per violation |
| **Solvency Margin** | Maintain 150% of required solvency | License suspension |
| **Investment Guidelines** | Asset allocation limits, approved investments | Financial penalties |
| **Claims Settlement** | 30-day settlement timeline | Consumer court cases |
| **Anti-Money Laundering** | KYC, suspicious transaction reporting | Criminal prosecution |

### MAS (Monetary Authority of Singapore)

MAS is Singapore's central bank and financial regulatory authority. For insurers:

| Area | Key Requirements | Penalty for Non-Compliance |
|------|-----------------|---------------------------|
| **Risk Management** | Enterprise Risk Management framework | Up to SGD 1 million |
| **Conduct Standards** | Fair dealing, disclosure requirements | License conditions |
| **Technology Risk** | Cybersecurity, business continuity | Regulatory action |
| **Outsourcing** | Due diligence, oversight of service providers | Fines |

### Compliance Mapping Strategy

Our system maps regulations to policies using:
1. **Semantic similarity**: Find policies that discuss similar topics
2. **Keyword matching**: Identify explicit regulatory references
3. **Gap detection**: Identify requirements without policy coverage

## 1.6 Data Discovery with Tracing

Now we discover what regulatory and policy documents are available in our dataset.

In [ ]:
# ============================================================================
# WEEK 1.6: DATA DISCOVERY FUNCTION
# ============================================================================

def discover_compliance_documents(
    regulations_dir: Path,
    policies_dir: Path
) -> Dict:
    """
    Discover all regulatory and policy documents with full tracing.

    This function scans the data directories and catalogs all available
    documents for processing.

    Args:
        regulations_dir: Path to regulatory PDFs
        policies_dir: Path to policy DOCX files

    Returns:
        Dict with 'regulations' and 'policies' lists
    """
    # Create trace for discovery operation
    trace = langfuse.trace(
        name="document-discovery",
        session_id=SESSION_ID,
        tags=["discovery", "documents", "setup"]
    )

    discovered = {
        'regulations': [],
        'policies': [],
        'other': []
    }

    # Span for scanning regulations
    span_regs = trace.span(name="scan-regulations-directory")
    if regulations_dir.exists():
        for f in sorted(regulations_dir.glob('*.pdf')):
            discovered['regulations'].append(f)
    span_regs.end(output={"pdf_files_found": len(discovered['regulations'])})

    # Span for scanning policies
    span_pols = trace.span(name="scan-policies-directory")
    if policies_dir.exists():
        for f in sorted(policies_dir.glob('*.docx')):
            discovered['policies'].append(f)
    span_pols.end(output={"docx_files_found": len(discovered['policies'])})

    # Update trace with summary
    trace.update(output={
        "total_regulations": len(discovered['regulations']),
        "total_policies": len(discovered['policies']),
        "regulations_dir_exists": regulations_dir.exists(),
        "policies_dir_exists": policies_dir.exists()
    })

    return discovered

print("discover_compliance_documents() function defined")

In [ ]:
# ============================================================================
# EXECUTE DATA DISCOVERY
# ============================================================================

compliance_files = discover_compliance_documents(REGULATIONS_DIR, POLICIES_DIR)

print("\nREGULATORY COMPLIANCE DOCUMENTS DISCOVERED")
print("=" * 60)

print(f"\nREGULATIONS (PDFs): {len(compliance_files['regulations'])}")
print("-" * 40)
for i, f in enumerate(compliance_files['regulations'], 1):
    print(f"  {i:2}. {f.name}")

print(f"\nCOMPANY POLICIES (DOCX): {len(compliance_files['policies'])}")
print("-" * 40)
for i, f in enumerate(compliance_files['policies'], 1):
    print(f"  {i:2}. {f.name}")

langfuse.flush()

## Checkpoint: Week 1 Verification

Before proceeding to Week 2, verify you have completed the following:

In [ ]:
# ============================================================================
# WEEK 1 CHECKPOINT
# ============================================================================

print("WEEK 1 CHECKPOINT - VERIFICATION")
print("=" * 60)

checkpoints = {
    "Environment configured": IN_COLAB is not None,
    "API keys loaded": bool(os.environ.get('OPENAI_API_KEY')),
    "Langfuse connected": 'langfuse' in globals(),
    "Session ID created": bool(SESSION_ID),
    "Traced functions defined": all(f in globals() for f in ['traced_embedding', 'traced_completion']),
    "Taxonomy defined": len(COMPLIANCE_DOMAINS) > 0,
    "Documents discovered": len(compliance_files.get('regulations', [])) > 0
}

all_passed = True
for check, passed in checkpoints.items():
    status = "PASS" if passed else "FAIL"
    symbol = "[x]" if passed else "[ ]"
    print(f"  {symbol} {check}: {status}")
    if not passed:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("Week 1 COMPLETE - Ready to proceed to Week 2!")
else:
    print("WARNING: Some checkpoints failed. Review before proceeding.")

# Log checkpoint to Langfuse
checkpoint_trace = langfuse.trace(
    name="week1-checkpoint",
    session_id=SESSION_ID,
    tags=["checkpoint", "week1"]
)
checkpoint_trace.update(output={"checkpoints": checkpoints, "all_passed": all_passed})
langfuse.flush()

## Week 1 Summary

### What We Accomplished

| Component | Status | Langfuse Traces |
|-----------|--------|----------------|
| Environment Setup | Complete | N/A |
| Langfuse Initialization | Complete | Session created |
| Traced Wrappers | Complete | test-embedding-*, test-completion-* |
| Compliance Taxonomy | Complete | compliance-taxonomy-definition |
| Document Discovery | Complete | document-discovery |

### Key Learnings

1. **Observability First**: Initialize Langfuse before any AI operations
2. **Session Organization**: Use SESSION_ID to group related traces
3. **Regulatory Landscape**: IRDAI and MAS have distinct but overlapping requirements
4. **Domain Taxonomy**: Compliance can be categorized into 9 key domains

### Next Week Preview

In Week 2, we will:
- Parse regulatory PDFs and extract text
- Parse company policy DOCX files
- Extract specific compliance requirements using NLP
- Perform exploratory data analysis on the corpus

---
# WEEK 2: Document Processing & Requirement Extraction (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 2: Document Processing & Requirement Extraction

---

## Learning Objectives

By the end of Week 2, you will be able to:

- [ ] Parse regulatory PDF documents and extract text
- [ ] Parse company policy DOCX files
- [ ] Extract specific compliance requirements using pattern matching
- [ ] Categorize requirements by compliance domain
- [ ] Perform exploratory data analysis on the compliance corpus

## 2.1 Regulatory Document Processor

This class handles parsing of both regulatory PDFs and company policy DOCX files.

In [ ]:
# ============================================================================
# WEEK 2.1: DOCUMENT PROCESSOR CLASS
# ============================================================================

import pdfplumber
from docx import Document as DocxDocument

class RegulatoryDocumentProcessor:
    """
    Process regulatory PDFs and company policy documents.
    All operations are traced to Langfuse for complete audit trail.
    """

    def __init__(self):
        self.processed_docs = []
        self.processing_errors = []

    def load_pdf(self, file_path: Path) -> Optional[Dict]:
        """Extract text and tables from a regulatory PDF."""
        trace = langfuse.trace(
            name="load-pdf",
            session_id=SESSION_ID,
            tags=["document", "pdf", "extraction"],
            metadata={"filename": file_path.name}
        )

        try:
            span = trace.span(name="extract-pdf-content")
            text_content = []
            tables = []

            with pdfplumber.open(file_path) as pdf:
                for page_num, page in enumerate(pdf.pages, 1):
                    text = page.extract_text()
                    if text:
                        text_content.append(text)
                    page_tables = page.extract_tables()
                    if page_tables:
                        for table in page_tables:
                            tables.append({'page': page_num, 'data': table})

            result = {
                'filename': file_path.name,
                'filepath': str(file_path),
                'format': 'pdf',
                'doc_type': 'regulation',
                'full_text': '\\n'.join(text_content),
                'tables': tables,
                'page_count': len(text_content),
                'char_count': sum(len(t) for t in text_content),
                'word_count': sum(len(t.split()) for t in text_content),
                'table_count': len(tables),
                'processed_at': datetime.now().isoformat()
            }

            span.end(output={"pages": result['page_count'], "words": result['word_count']})
            trace.update(output={"status": "success", "filename": file_path.name})
            return result

        except Exception as e:
            trace.update(output={"status": "error", "error": str(e)})
            self.processing_errors.append({'file': file_path.name, 'error': str(e)})
            print(f"Error loading PDF {file_path.name}: {e}")
            return None

print("RegulatoryDocumentProcessor.load_pdf() defined")

In [ ]:
# ============================================================================
# DOCX LOADER METHOD
# ============================================================================

def load_docx(self, file_path: Path) -> Optional[Dict]:
    """Extract text and tables from a company policy DOCX file."""
    trace = langfuse.trace(
        name="load-docx",
        session_id=SESSION_ID,
        tags=["document", "docx", "extraction"],
        metadata={"filename": file_path.name}
    )

    try:
        span = trace.span(name="extract-docx-content")
        doc = DocxDocument(file_path)

        paragraphs = []
        sections = []
        current_section = None

        for para in doc.paragraphs:
            text = para.text.strip()
            if not text:
                continue
            paragraphs.append(text)
            if para.style.name.startswith('Heading'):
                current_section = text
                sections.append({'title': text, 'content': []})
            elif current_section and sections:
                sections[-1]['content'].append(text)

        tables = []
        for table in doc.tables:
            table_data = []
            for row in table.rows:
                row_data = [cell.text.strip() for cell in row.cells]
                if any(row_data):
                    table_data.append(row_data)
            if table_data:
                tables.append(table_data)

        full_text = '\\n'.join(paragraphs)

        result = {
            'filename': file_path.name,
            'filepath': str(file_path),
            'format': 'docx',
            'doc_type': 'policy',
            'full_text': full_text,
            'sections': sections,
            'tables': tables,
            'paragraph_count': len(paragraphs),
            'section_count': len(sections),
            'char_count': len(full_text),
            'word_count': len(full_text.split()),
            'table_count': len(tables),
            'processed_at': datetime.now().isoformat()
        }

        span.end(output={"sections": result['section_count'], "words": result['word_count']})
        trace.update(output={"status": "success", "filename": file_path.name})
        return result

    except Exception as e:
        trace.update(output={"status": "error", "error": str(e)})
        self.processing_errors.append({'file': file_path.name, 'error': str(e)})
        print(f"Error loading DOCX {file_path.name}: {e}")
        return None

RegulatoryDocumentProcessor.load_docx = load_docx
print("RegulatoryDocumentProcessor.load_docx() defined")

In [ ]:
# ============================================================================
# BATCH PROCESSING METHOD
# ============================================================================

def process_all_documents(self, files: Dict) -> Dict:
    """Process all regulatory and policy documents."""
    trace = langfuse.trace(
        name="process-all-documents",
        session_id=SESSION_ID,
        tags=["processing", "batch", "documents"]
    )

    processed = {'regulations': [], 'policies': []}

    span_regs = trace.span(name="process-regulations")
    for pdf_path in files.get('regulations', []):
        doc = self.load_pdf(pdf_path)
        if doc:
            doc['doc_id'] = f"REG-{len(processed['regulations'])+1:03d}"
            processed['regulations'].append(doc)
    span_regs.end(output={"processed": len(processed['regulations'])})

    span_pols = trace.span(name="process-policies")
    for docx_path in files.get('policies', []):
        doc = self.load_docx(docx_path)
        if doc:
            doc['doc_id'] = f"POL-{len(processed['policies'])+1:03d}"
            processed['policies'].append(doc)
    span_pols.end(output={"processed": len(processed['policies'])})

    trace.update(output={"regulations_processed": len(processed['regulations']), "policies_processed": len(processed['policies'])})
    return processed

RegulatoryDocumentProcessor.process_all_documents = process_all_documents
print("RegulatoryDocumentProcessor.process_all_documents() defined")

In [ ]:
# ============================================================================
# EXECUTE DOCUMENT PROCESSING
# ============================================================================

doc_processor = RegulatoryDocumentProcessor()

print("Processing compliance documents...")
print("=" * 60)

all_compliance_docs = doc_processor.process_all_documents(compliance_files)

print(f"\\nProcessing Complete!")
print(f"  Regulations processed: {len(all_compliance_docs['regulations'])}")
print(f"  Policies processed: {len(all_compliance_docs['policies'])}")

langfuse.flush()

## Key Concept: Document Structure in Compliance

Understanding document structure is crucial for accurate requirement extraction:

### Regulatory Documents (PDFs)

| Component | Purpose | Example |
|-----------|---------|--------|
| **Preamble** | Context and authority | "In exercise of powers conferred by Section 14..." |
| **Definitions** | Key term definitions | "'Insurer' means a company registered under..." |
| **Requirements** | Mandatory obligations | "The insurer shall maintain..." |
| **Timelines** | Compliance deadlines | "Within 30 days of the end of each quarter..." |
| **Penalties** | Non-compliance consequences | "Failure to comply shall attract a penalty..." |

### Linguistic Patterns for Requirements

Regulatory requirements often use specific modal verbs:
- **"shall"**: Mandatory requirement
- **"must"**: Absolute requirement
- **"should"**: Recommended practice
- **"may"**: Optional/permitted

## 2.2 Exploratory Data Analysis (EDA)

Let's analyze the characteristics of our compliance document corpus.

In [ ]:
# ============================================================================
# WEEK 2.2: CREATE EDA DATAFRAMES
# ============================================================================

import matplotlib.pyplot as plt
import seaborn as sns

eda_trace = langfuse.trace(
    name="compliance-eda",
    session_id=SESSION_ID,
    tags=["eda", "analysis", "visualization"]
)

reg_data = []
for doc in all_compliance_docs['regulations']:
    reg_data.append({
        'doc_id': doc['doc_id'],
        'filename': doc['filename'][:50],
        'pages': doc.get('page_count', 0),
        'chars': doc.get('char_count', 0),
        'words': doc.get('word_count', 0),
        'tables': doc.get('table_count', 0),
        'type': 'Regulation'
    })

pol_data = []
for doc in all_compliance_docs['policies']:
    pol_data.append({
        'doc_id': doc['doc_id'],
        'filename': doc['filename'][:50],
        'sections': doc.get('section_count', 0),
        'paragraphs': doc.get('paragraph_count', 0),
        'chars': doc.get('char_count', 0),
        'words': doc.get('word_count', 0),
        'tables': doc.get('table_count', 0),
        'type': 'Policy'
    })

df_regs = pd.DataFrame(reg_data)
df_pols = pd.DataFrame(pol_data)

print("DataFrames created for EDA")
print(f"  Regulations: {len(df_regs)} rows")
print(f"  Policies: {len(df_pols)} rows")

In [ ]:
# ============================================================================
# REGULATIONS ANALYSIS
# ============================================================================

print("REGULATORY DOCUMENTS ANALYSIS")
print("=" * 70)

if not df_regs.empty:
    print(df_regs[['doc_id', 'filename', 'pages', 'words', 'tables']].to_string(index=False))
    print("\\nStatistics:")
    print(f"  Total documents: {len(df_regs)}")
    print(f"  Total pages: {df_regs['pages'].sum()}")
    print(f"  Total words: {df_regs['words'].sum():,}")
    print(f"  Average words per document: {df_regs['words'].mean():,.0f}")
else:
    print("No regulations found in the dataset.")

In [ ]:
# ============================================================================
# POLICIES ANALYSIS
# ============================================================================

print("POLICY DOCUMENTS ANALYSIS")
print("=" * 70)

if not df_pols.empty:
    print(df_pols[['doc_id', 'filename', 'sections', 'words', 'tables']].to_string(index=False))
    print("\\nStatistics:")
    print(f"  Total documents: {len(df_pols)}")
    print(f"  Total sections: {df_pols['sections'].sum()}")
    print(f"  Total words: {df_pols['words'].sum():,}")
else:
    print("No policies found in the dataset.")

In [ ]:
# ============================================================================
# EDA VISUALIZATIONS
# ============================================================================

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: Document Counts by Type
ax1 = axes[0, 0]
doc_types = ['Regulations (PDF)', 'Policies (DOCX)']
doc_counts = [len(all_compliance_docs['regulations']), len(all_compliance_docs['policies'])]
colors = ['#e74c3c', '#3498db']
bars = ax1.bar(doc_types, doc_counts, color=colors, edgecolor='black')
ax1.set_title('Document Count by Type', fontsize=12, fontweight='bold')
ax1.set_ylabel('Count')
for bar, count in zip(bars, doc_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.2, str(count), ha='center', fontweight='bold')

# Plot 2: Word Count Distribution (Regulations)
ax2 = axes[0, 1]
if not df_regs.empty:
    ax2.barh(df_regs['filename'].str[:30], df_regs['words'], color='#e74c3c', alpha=0.7)
    ax2.set_xlabel('Word Count')
    ax2.set_title('Regulation Document Sizes', fontsize=12, fontweight='bold')
else:
    ax2.text(0.5, 0.5, 'No regulations found', ha='center', va='center')

# Plot 3: Word Count Distribution (Policies)
ax3 = axes[1, 0]
if not df_pols.empty:
    ax3.barh(df_pols['filename'].str[:30], df_pols['words'], color='#3498db', alpha=0.7)
    ax3.set_xlabel('Word Count')
    ax3.set_title('Policy Document Sizes', fontsize=12, fontweight='bold')
else:
    ax3.text(0.5, 0.5, 'No policies found', ha='center', va='center')

# Plot 4: Domain Coverage
ax4 = axes[1, 1]
domains = list(COMPLIANCE_DOMAINS.keys())
reg_counts = [len(COMPLIANCE_DOMAINS[d]['regulations']) for d in domains]
pol_counts = [len(COMPLIANCE_DOMAINS[d]['policies']) for d in domains]
x = np.arange(len(domains))
width = 0.35
ax4.bar(x - width/2, reg_counts, width, label='Regulations', color='#e74c3c', alpha=0.7)
ax4.bar(x + width/2, pol_counts, width, label='Policies', color='#3498db', alpha=0.7)
ax4.set_xticks(x)
ax4.set_xticklabels([d[:8] for d in domains], rotation=45, ha='right')
ax4.set_title('Domain Coverage', fontsize=12, fontweight='bold')
ax4.legend()

plt.tight_layout()
plt.savefig('compliance_eda.png', dpi=150, bbox_inches='tight')
plt.show()

print("\\nEDA visualizations saved to compliance_eda.png")
langfuse.flush()

## 2.3 Requirement Extractor

This component extracts specific compliance requirements from regulatory documents using pattern matching.

In [ ]:
# ============================================================================
# WEEK 2.3: REQUIREMENT EXTRACTOR CLASS
# ============================================================================

class RequirementExtractor:
    """
    Extract regulatory requirements from documents.
    Uses pattern matching to identify mandatory, prohibited, and recommended actions.
    """

    def __init__(self):
        self.requirement_patterns = [
            r'(?:the\\s+)?(?:insurer|company|reinsurer|board|committee)\\s+(?:shall|must)\\s+(.+?)(?:\\.|;|$)',
            r'(?:it is mandatory|mandatory requirement)\\s*:?\\s*(.+?)(?:\\.|;|$)',
            r'(?:shall|must)\\s+(?:ensure|maintain|establish|implement|report|submit)\\s+(.+?)(?:\\.|;|$)',
            r'(?:shall not|must not|is prohibited from)\\s+(.+?)(?:\\.|;|$)',
            r'(?:should|is recommended to)\\s+(.+?)(?:\\.|;|$)',
        ]

        self.domain_keywords = {
            'governance': ['board', 'director', 'governance', 'corporate', 'committee', 'chairman'],
            'risk_management': ['risk', 'assessment', 'mitigation', 'exposure', 'tolerance'],
            'information_security': ['security', 'cyber', 'data', 'privacy', 'breach', 'encryption'],
            'audit': ['audit', 'control', 'internal', 'review', 'assurance'],
            'fraud': ['fraud', 'misrepresentation', 'investigation', 'suspicious'],
            'reinsurance': ['reinsurance', 'cede', 'retrocession', 'treaty'],
            'claims': ['claim', 'settlement', 'grievance', 'redressal'],
            'investments': ['investment', 'asset', 'portfolio', 'solvency'],
            'sales': ['sales', 'distribution', 'agent', 'broker', 'commission']
        }

    def extract_requirements(self, text: str, doc_id: str = None) -> List[Dict]:
        """Extract requirement statements from regulatory text."""
        trace = langfuse.trace(
            name="extract-requirements",
            session_id=SESSION_ID,
            tags=["extraction", "requirements"],
            metadata={"doc_id": doc_id}
        )

        requirements = []
        seen = set()

        for pattern in self.requirement_patterns:
            matches = re.findall(pattern, text, re.IGNORECASE | re.MULTILINE)
            for match in matches:
                req_text = match if isinstance(match, str) else ' '.join(match)
                req_text = req_text.strip()

                if 20 < len(req_text) < 500 and req_text not in seen:
                    seen.add(req_text)
                    category = self._categorize_requirement(req_text)
                    req_type = self._determine_type(req_text)
                    requirements.append({
                        'text': req_text,
                        'category': category,
                        'type': req_type,
                        'source_doc': doc_id
                    })

        requirements = requirements[:50]
        trace.update(output={"requirements_found": len(requirements)})
        return requirements

    def _categorize_requirement(self, text: str) -> str:
        text_lower = text.lower()
        for domain, keywords in self.domain_keywords.items():
            if any(kw in text_lower for kw in keywords):
                return domain
        return 'general'

    def _determine_type(self, text: str) -> str:
        text_lower = text.lower()
        if any(phrase in text_lower for phrase in ['shall not', 'must not', 'prohibited']):
            return 'prohibition'
        elif any(phrase in text_lower for phrase in ['shall', 'must', 'mandatory']):
            return 'mandatory'
        elif 'should' in text_lower:
            return 'recommended'
        return 'statement'

print("RequirementExtractor class defined")

In [ ]:
# ============================================================================
# EXAMPLE 1: Single Document Requirement Extraction
# ============================================================================

req_extractor = RequirementExtractor()

print("EXAMPLE 1: Single Document Requirement Extraction")
print("=" * 60)

if all_compliance_docs['regulations']:
    sample_doc = all_compliance_docs['regulations'][0]
    print(f"Document: {sample_doc['filename']}")

    requirements = req_extractor.extract_requirements(sample_doc['full_text'], sample_doc['doc_id'])
    print(f"Requirements extracted: {len(requirements)}\\n")

    for i, req in enumerate(requirements[:5], 1):
        print(f"{i}. [{req['type'].upper()}] [{req['category']}]")
        print(f"   {req['text'][:100]}...\\n")
else:
    print("No regulations available.")

langfuse.flush()

In [ ]:
# ============================================================================
# EXAMPLE 2: Multi-Document Extraction
# ============================================================================

print("EXAMPLE 2: Multi-Document Requirement Extraction")
print("=" * 60)

all_requirements = []

for doc in all_compliance_docs['regulations'][:5]:
    reqs = req_extractor.extract_requirements(doc['full_text'], doc['doc_id'])
    all_requirements.extend(reqs)

    print(f"{doc['doc_id']}: {doc['filename'][:40]}...")
    print(f"  Requirements: {len(reqs)}")

    categories = {}
    for req in reqs:
        cat = req['category']
        categories[cat] = categories.get(cat, 0) + 1
    for cat, count in sorted(categories.items()):
        print(f"    - {cat}: {count}")
    print()

print(f"\\nTotal requirements extracted: {len(all_requirements)}")
langfuse.flush()

In [ ]:
# ============================================================================
# EXAMPLE 3: Requirement Distribution Analysis
# ============================================================================

print("EXAMPLE 3: Requirement Distribution Analysis")
print("=" * 60)

category_counts = {}
type_counts = {}

for req in all_requirements:
    cat = req['category']
    typ = req['type']
    category_counts[cat] = category_counts.get(cat, 0) + 1
    type_counts[typ] = type_counts.get(typ, 0) + 1

print("\\nBy Compliance Domain:")
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    if all_requirements:
        pct = (count / len(all_requirements)) * 100
        bar = '=' * int(pct / 2)
        print(f"  {cat:20} {count:4} ({pct:5.1f}%) {bar}")

print("\\nBy Requirement Type:")
for typ, count in sorted(type_counts.items(), key=lambda x: -x[1]):
    if all_requirements:
        pct = (count / len(all_requirements)) * 100
        print(f"  {typ:15} {count:4} ({pct:5.1f}%)")

## Checkpoint: Week 2 Verification

In [ ]:
# ============================================================================
# WEEK 2 CHECKPOINT
# ============================================================================

print("WEEK 2 CHECKPOINT - VERIFICATION")
print("=" * 60)

checkpoints = {
    "Document processor defined": 'RegulatoryDocumentProcessor' in globals(),
    "Regulations processed": len(all_compliance_docs.get('regulations', [])) > 0,
    "Policies processed": len(all_compliance_docs.get('policies', [])) > 0,
    "EDA completed": 'df_regs' in globals() and 'df_pols' in globals(),
    "Requirement extractor defined": 'RequirementExtractor' in globals(),
    "Requirements extracted": len(all_requirements) > 0
}

all_passed = True
for check, passed in checkpoints.items():
    status = "PASS" if passed else "FAIL"
    symbol = "[x]" if passed else "[ ]"
    print(f"  {symbol} {check}: {status}")
    if not passed:
        all_passed = False

print("\\n" + "=" * 60)
if all_passed:
    print("Week 2 COMPLETE - Ready to proceed to Week 3!")
else:
    print("WARNING: Some checkpoints failed.")

checkpoint_trace = langfuse.trace(name="week2-checkpoint", session_id=SESSION_ID, tags=["checkpoint", "week2"])
checkpoint_trace.update(output={"checkpoints": checkpoints, "all_passed": all_passed})
langfuse.flush()

## Week 2 Summary

### What We Accomplished

| Component | Status | Key Output |
|-----------|--------|------------|
| PDF Processing | Complete | Extracted text and tables from regulations |
| DOCX Processing | Complete | Extracted content from policies |
| EDA | Complete | Corpus statistics and visualizations |
| Requirement Extraction | Complete | Categorized requirements by domain |

### Key Learnings

1. **Document Structure**: Regulatory documents have specific patterns (shall/must/should)
2. **Pattern Matching**: Regular expressions effectively extract requirements
3. **Domain Categorization**: Keywords map requirements to compliance domains

### Next Week Preview

In Week 3, we will:
- Set up ChromaDB vector store
- Generate embeddings for regulations and policies
- Implement semantic search for compliance matching

---
# WEEK 3: ChromaDB Vector Store & Semantic Search (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 3: ChromaDB Vector Store & Semantic Search

---

## Learning Objectives

By the end of Week 3, you will be able to:

- [ ] Understand how embeddings represent semantic meaning
- [ ] Set up ChromaDB with persistent storage
- [ ] Implement document chunking strategies for compliance documents
- [ ] Create a fully traced vector store for regulations and policies
- [ ] Perform semantic search to find matching policies for regulations

## Key Concept: Why Vector Search for Compliance?

Traditional keyword search fails in compliance because:
- Regulations say "board shall meet quarterly" but policies say "quarterly board meetings"
- Different terminology: "cybersecurity" vs "information security" vs "data protection"
- Semantic relationships: "fraud prevention" relates to "suspicious activity monitoring"

**Vector embeddings capture semantic meaning**, allowing us to:
1. Find policies that address regulatory requirements even with different wording
2. Identify gaps where no policy matches a regulation
3. Cluster related requirements across different documents

### Embedding Similarity Interpretation

| Cosine Similarity | Interpretation | Compliance Action |
|-------------------|----------------|-------------------|
| > 0.85 | Very high match | Policy likely addresses requirement |
| 0.70 - 0.85 | Good match | Policy partially addresses requirement |
| 0.55 - 0.70 | Related content | Review for potential coverage |
| < 0.55 | Weak match | Likely a gap - no policy coverage |

## 3.1 Compliance Vector Store Class

In [ ]:
# ============================================================================
# WEEK 3.1: COMPLIANCE VECTOR STORE
# ============================================================================

import chromadb

class ComplianceVectorStore:
    """
    ChromaDB vector store for regulatory compliance documents.
    All operations are traced to Langfuse for complete observability.

    This store enables:
    - Semantic search across regulations and policies
    - Policy-to-regulation matching
    - Gap detection through similarity analysis
    """

    def __init__(self, persist_directory: str = "./chroma_compliance_db"):
        trace = langfuse.trace(
            name="init-vector-store",
            session_id=SESSION_ID,
            tags=["vectorstore", "chromadb", "init"]
        )

        self.embedding_model = "text-embedding-3-small"
        self.client = chromadb.PersistentClient(path=persist_directory)
        self.collections = {}

        trace.update(output={"status": "initialized", "path": persist_directory})
        print(f"Compliance Vector Store initialized at {persist_directory}")

    def get_embedding(self, text: str, doc_id: str = None) -> List[float]:
        """Generate embedding with tracing."""
        return traced_embedding(
            text,
            trace_name="vectorstore-embedding",
            metadata={"doc_id": doc_id, "text_length": len(text)}
        )

    def create_collection(self, name: str) -> chromadb.Collection:
        """Create or get a collection for compliance documents."""
        collection = self.client.get_or_create_collection(
            name=name,
            metadata={"hnsw:space": "cosine"}
        )
        self.collections[name] = collection
        return collection

    def chunk_document(self, text: str, chunk_size: int = 1000, overlap: int = 100) -> List[str]:
        """
        Split document into overlapping chunks for better retrieval.

        Args:
            text: Full document text
            chunk_size: Target words per chunk
            overlap: Words to overlap between chunks

        Returns:
            List of text chunks
        """
        words = text.split()
        chunks = []

        for i in range(0, len(words), chunk_size - overlap):
            chunk = ' '.join(words[i:i + chunk_size])
            if chunk.strip():
                chunks.append(chunk)

        return chunks if chunks else [text]

print("ComplianceVectorStore base class defined")

In [ ]:
# ============================================================================
# VECTOR STORE: ADD DOCUMENTS METHOD
# ============================================================================

def add_documents(self, collection_name: str, documents: List[Dict], doc_type: str):
    """
    Add compliance documents to vector store with full tracing.

    Args:
        collection_name: Name of the ChromaDB collection
        documents: List of document dictionaries with 'full_text' and 'doc_id'
        doc_type: Type of document ('regulation' or 'policy')
    """
    trace = langfuse.trace(
        name="add-documents-to-vectorstore",
        session_id=SESSION_ID,
        tags=["vectorstore", "indexing", doc_type]
    )

    collection = self.create_collection(collection_name)

    ids, embeddings, texts, metadatas = [], [], [], []

    for doc in documents:
        span = trace.span(name=f"index-{doc['doc_id']}")

        # Chunk the document
        chunks = self.chunk_document(doc.get('full_text', ''))

        for i, chunk in enumerate(chunks):
            chunk_id = f"{doc['doc_id']}_chunk_{i}"

            # Generate embedding
            embedding = self.get_embedding(chunk, doc['doc_id'])

            ids.append(chunk_id)
            embeddings.append(embedding)
            texts.append(chunk)
            metadatas.append({
                'doc_id': doc['doc_id'],
                'filename': doc['filename'],
                'doc_type': doc_type,
                'chunk_index': i,
                'total_chunks': len(chunks)
            })

        span.end(output={"chunks": len(chunks)})

    # Add to collection
    if ids:
        collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=texts,
            metadatas=metadatas
        )

    trace.update(output={
        "documents_indexed": len(documents),
        "chunks_created": len(ids),
        "doc_type": doc_type
    })

    print(f"Added {len(ids)} chunks from {len(documents)} {doc_type}s to '{collection_name}'")

ComplianceVectorStore.add_documents = add_documents
print("ComplianceVectorStore.add_documents() defined")

In [ ]:
# ============================================================================
# VECTOR STORE: SEARCH METHOD
# ============================================================================

def search(self, collection_name: str, query: str, n_results: int = 5,
           doc_type_filter: str = None) -> Dict:
    """
    Semantic search with full tracing.

    Args:
        collection_name: Name of the collection to search
        query: Search query text
        n_results: Number of results to return
        doc_type_filter: Optional filter for 'regulation' or 'policy'

    Returns:
        Dict with documents, metadatas, and distances
    """
    trace = langfuse.trace(
        name="semantic-search",
        session_id=SESSION_ID,
        tags=["search", "vectorstore"],
        input={"query": query[:200], "n_results": n_results, "filter": doc_type_filter}
    )

    if collection_name not in self.collections:
        self.create_collection(collection_name)

    collection = self.collections[collection_name]

    # Generate query embedding
    query_embedding = self.get_embedding(query, "query")

    # Build filter
    where_filter = {"doc_type": doc_type_filter} if doc_type_filter else None

    # Execute search
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        where=where_filter,
        include=['documents', 'metadatas', 'distances']
    )

    # Log results
    trace.update(output={
        "results_found": len(results.get('ids', [[]])[0]),
        "top_similarity": 1 - results['distances'][0][0] if results.get('distances') and results['distances'][0] else None
    })

    return results

ComplianceVectorStore.search = search
print("ComplianceVectorStore.search() defined")

In [ ]:
# ============================================================================
# VECTOR STORE: SPECIALIZED SEARCH METHODS
# ============================================================================

def find_matching_policies(self, regulation_text: str, n_results: int = 3) -> Dict:
    """Find policies that match a regulatory requirement."""
    return self.search('compliance_all', regulation_text, n_results, 'policy')

def find_relevant_regulations(self, policy_topic: str, n_results: int = 3) -> Dict:
    """Find regulations relevant to a policy topic."""
    return self.search('compliance_all', policy_topic, n_results, 'regulation')

def find_similar_requirements(self, requirement_text: str, n_results: int = 5) -> Dict:
    """Find similar requirements across all documents."""
    return self.search('compliance_all', requirement_text, n_results)

ComplianceVectorStore.find_matching_policies = find_matching_policies
ComplianceVectorStore.find_relevant_regulations = find_relevant_regulations
ComplianceVectorStore.find_similar_requirements = find_similar_requirements

print("Specialized search methods defined")

## 3.2 Initialize and Index Documents

In [ ]:
# ============================================================================
# WEEK 3.2: INITIALIZE VECTOR STORE
# ============================================================================

# Initialize vector store
compliance_vector_store = ComplianceVectorStore()

print("Compliance Vector Store ready for indexing")

In [ ]:
# ============================================================================
# INDEX REGULATIONS
# ============================================================================

print("Indexing regulations...")
print("=" * 50)

compliance_vector_store.add_documents(
    'compliance_all',
    all_compliance_docs['regulations'],
    'regulation'
)

langfuse.flush()
print("\nRegulations indexed!")

In [ ]:
# ============================================================================
# INDEX POLICIES
# ============================================================================

print("Indexing policies...")
print("=" * 50)

compliance_vector_store.add_documents(
    'compliance_all',
    all_compliance_docs['policies'],
    'policy'
)

langfuse.flush()
print("\nPolicies indexed!")
print("\nAll compliance documents indexed to vector store!")

## 3.3 Semantic Search Examples

In [ ]:
# ============================================================================
# EXAMPLE 1: Find Policies for Governance Requirements
# ============================================================================

print("EXAMPLE 1: Find Policies for Governance Requirements")
print("=" * 60)

query = "board of directors governance requirements and independent directors"
print(f"Query: '{query}'\n")

results = compliance_vector_store.find_matching_policies(query, 3)

if results['documents'] and results['documents'][0]:
    print("Matching Policies:")
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ), 1):
        similarity = 1 - dist
        print(f"\n{i}. {meta['filename']} (similarity: {similarity:.3f})")
        print(f"   Chunk {meta['chunk_index']+1} of {meta['total_chunks']}")
        print(f"   Preview: {doc[:200]}...")
else:
    print("No matching policies found.")

langfuse.flush()

In [ ]:
# ============================================================================
# EXAMPLE 2: Find Regulations for Cybersecurity
# ============================================================================

print("EXAMPLE 2: Find Regulations for Cybersecurity")
print("=" * 60)

query = "cybersecurity controls data protection encryption requirements"
print(f"Query: '{query}'\n")

results = compliance_vector_store.find_relevant_regulations(query, 3)

if results['documents'] and results['documents'][0]:
    print("Relevant Regulations:")
    for i, (doc, meta, dist) in enumerate(zip(
        results['documents'][0],
        results['metadatas'][0],
        results['distances'][0]
    ), 1):
        similarity = 1 - dist
        print(f"\n{i}. {meta['filename']} (similarity: {similarity:.3f})")
        print(f"   Preview: {doc[:200]}...")
else:
    print("No relevant regulations found.")

langfuse.flush()

In [ ]:
# ============================================================================
# EXAMPLE 3: Multiple Compliance Queries
# ============================================================================

print("EXAMPLE 3: Multiple Compliance Queries")
print("=" * 60)

test_queries = [
    "fraud prevention and detection mechanisms",
    "reinsurance agreement requirements",
    "internal audit procedures and controls",
    "claims settlement timeline requirements",
    "risk management framework implementation"
]

for i, query in enumerate(test_queries, 1):
    print(f"\n[Query {i}] '{query}'")

    # Search regulations
    reg_results = compliance_vector_store.find_relevant_regulations(query, 2)
    print("  Regulations:")
    if reg_results['documents'] and reg_results['documents'][0]:
        for meta, dist in zip(reg_results['metadatas'][0], reg_results['distances'][0]):
            print(f"    - {meta['filename'][:40]} (sim: {1-dist:.3f})")

    # Search policies
    pol_results = compliance_vector_store.find_matching_policies(query, 2)
    print("  Policies:")
    if pol_results['documents'] and pol_results['documents'][0]:
        for meta, dist in zip(pol_results['metadatas'][0], pol_results['distances'][0]):
            print(f"    - {meta['filename'][:40]} (sim: {1-dist:.3f})")

langfuse.flush()

## Deep Dive: Understanding Similarity Scores in Compliance

ChromaDB uses cosine distance, which we convert to similarity (1 - distance):

### Interpreting Scores for Compliance Decisions

| Similarity | Interpretation | Compliance Action |
|------------|----------------|-------------------|
| > 0.85 | Very high match | Policy directly addresses requirement |
| 0.70 - 0.85 | Good match | Policy likely addresses requirement - verify |
| 0.55 - 0.70 | Partial match | Policy may partially address - gap possible |
| < 0.55 | Weak match | Likely compliance gap - needs attention |

### Why These Thresholds?

1. **> 0.85**: Same topic, similar language (e.g., both discuss "board meetings")
2. **0.70-0.85**: Related topic, different phrasing (e.g., "governance" vs "corporate structure")
3. **0.55-0.70**: Tangentially related (e.g., "risk management" vs "compliance monitoring")
4. **< 0.55**: Different topics entirely

### Calibration Tip

These thresholds should be calibrated based on your specific corpus. Run test queries with known answers to validate.

## Checkpoint: Week 3 Verification

In [ ]:
# ============================================================================
# WEEK 3 CHECKPOINT
# ============================================================================

print("WEEK 3 CHECKPOINT - VERIFICATION")
print("=" * 60)

checkpoints = {
    "Vector store initialized": 'compliance_vector_store' in globals(),
    "Collection created": 'compliance_all' in compliance_vector_store.collections,
    "Regulations indexed": True,  # We indexed above
    "Policies indexed": True,
    "Search working": True  # Examples ran above
}

all_passed = True
for check, passed in checkpoints.items():
    status = "PASS" if passed else "FAIL"
    symbol = "[x]" if passed else "[ ]"
    print(f"  {symbol} {check}: {status}")
    if not passed:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("Week 3 COMPLETE - Ready to proceed to Week 4!")
else:
    print("WARNING: Some checkpoints failed.")

checkpoint_trace = langfuse.trace(name="week3-checkpoint", session_id=SESSION_ID, tags=["checkpoint", "week3"])
checkpoint_trace.update(output={"checkpoints": checkpoints, "all_passed": all_passed})
langfuse.flush()

## Week 3 Summary

### What We Accomplished

| Component | Status | Key Output |
|-----------|--------|------------|
| ChromaDB Setup | Complete | Persistent vector store |
| Document Chunking | Complete | Overlapping chunks for better retrieval |
| Embedding Generation | Complete | All documents embedded |
| Semantic Search | Complete | Policy-regulation matching |

### Key Learnings

1. **Chunking Strategy**: Overlapping chunks preserve context at boundaries
2. **Similarity Thresholds**: Calibrate based on your domain
3. **Filtered Search**: Filter by doc_type for targeted results

### Next Week Preview

In Week 4, we will:
- Design Pydantic models for structured compliance outputs
- Build specialized agents (Regulation Monitor, Gap Detector, Policy Analyzer)
- Implement LangChain LCEL chains with full tracing

---
# WEEK 4: Compliance Agent Design (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 4: Compliance Agent Design

---

## Learning Objectives

By the end of Week 4, you will be able to:

- [ ] Design Pydantic models for structured compliance outputs
- [ ] Build a Regulation Monitor Agent that analyzes regulatory requirements
- [ ] Build a Gap Detector Agent that identifies compliance gaps
- [ ] Build a Policy Analyzer Agent that scores policy compliance
- [ ] Implement LangChain LCEL chains with full Langfuse tracing

## Key Concept: Why Structured Outputs for Compliance?

In compliance, **consistency and auditability** are paramount. Free-form text responses are:
- Hard to aggregate across multiple assessments
- Difficult to audit and explain
- Inconsistent in format and terminology

**Pydantic models** enforce:
- Consistent output structure
- Type validation (scores are floats, not strings)
- Required fields (every assessment has a severity level)
- Enumerated values (severity is always HIGH/MEDIUM/LOW)

### Agent Architecture

```
                    [Compliance Assessment Request]
                              |
         +--------------------+--------------------+
         |                    |                    |
         v                    v                    v
   [Regulation         [Gap Detector       [Policy Analyzer
    Monitor Agent]      Agent]              Agent]
         |                    |                    |
         v                    v                    v
   RegulationAnalysis   GapAssessment      ComplianceScore
   (Pydantic)           (Pydantic)         (Pydantic)
```

## 4.1 Pydantic Models for Structured Output

In [ ]:
# ============================================================================
# WEEK 4.1: PYDANTIC MODELS
# ============================================================================

from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

class RegulationAnalysis(BaseModel):
    """
    Structured analysis of a regulatory requirement.
    Used by the Regulation Monitor Agent.
    """
    requirement_summary: str = Field(
        description="Concise summary of the regulatory requirement (1-2 sentences)"
    )
    compliance_domain: str = Field(
        description="Domain: governance, risk_management, information_security, audit, fraud, reinsurance, claims, investments, or sales"
    )
    mandatory_actions: List[str] = Field(
        description="List of specific actions required for compliance"
    )
    evidence_needed: List[str] = Field(
        description="List of evidence/documentation needed to demonstrate compliance"
    )
    risk_level: str = Field(
        description="Risk if non-compliant: LOW, MEDIUM, HIGH, or CRITICAL"
    )
    timeline: str = Field(
        description="Compliance timeline if specified, otherwise 'Not specified'",
        default="Not specified"
    )
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Confidence in this analysis (0.0 to 1.0)"
    )

print("RegulationAnalysis model defined")

In [ ]:
# ============================================================================
# GAP ASSESSMENT MODEL
# ============================================================================

class GapAssessment(BaseModel):
    """
    Assessment of a compliance gap between regulation and policy.
    Used by the Gap Detector Agent.
    """
    requirement: str = Field(
        description="The regulatory requirement being assessed"
    )
    current_state: str = Field(
        description="Current compliance state based on policy review"
    )
    gap_identified: bool = Field(
        description="Whether a compliance gap exists"
    )
    gap_description: str = Field(
        description="Detailed description of the gap (if any)"
    )
    severity: str = Field(
        description="Gap severity: LOW, MEDIUM, HIGH, or CRITICAL"
    )
    remediation_steps: List[str] = Field(
        description="Specific steps to remediate the gap"
    )
    estimated_effort: str = Field(
        description="Effort to remediate: Quick Fix, Moderate, Significant, Major Project"
    )
    confidence: float = Field(
        ge=0.0, le=1.0,
        description="Confidence in this assessment (0.0 to 1.0)"
    )

print("GapAssessment model defined")

In [ ]:
# ============================================================================
# COMPLIANCE SCORE MODEL
# ============================================================================

class ComplianceScore(BaseModel):
    """
    Overall compliance score for a domain.
    Used by the Policy Analyzer Agent.
    """
    domain: str = Field(
        description="Compliance domain being scored"
    )
    score: float = Field(
        ge=0.0, le=100.0,
        description="Compliance score from 0 to 100"
    )
    status: str = Field(
        description="Status: COMPLIANT, PARTIALLY_COMPLIANT, or NON_COMPLIANT"
    )
    gaps_count: int = Field(
        description="Number of gaps identified"
    )
    critical_gaps: List[str] = Field(
        description="List of critical gaps requiring immediate action"
    )
    recommendations: List[str] = Field(
        description="Improvement recommendations"
    )
    next_review_date: str = Field(
        description="Recommended next review date",
        default="Quarterly"
    )

print("ComplianceScore model defined")
print("\nAll Pydantic models ready for agent integration")

## 4.2 Regulation Monitor Agent

In [ ]:
# ============================================================================
# WEEK 4.2: REGULATION MONITOR AGENT
# ============================================================================

class RegulationMonitorAgent:
    """
    Agent for analyzing regulatory requirements.

    Responsibilities:
    - Parse regulatory text to extract requirements
    - Categorize requirements by compliance domain
    - Identify mandatory actions and evidence needed
    - Assess risk level of non-compliance

    All invocations are traced to Langfuse.
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model_name = model
        self.llm = ChatOpenAI(model=model, temperature=0)
        self.parser = PydanticOutputParser(pydantic_object=RegulationAnalysis)
        self.prompt = self._create_prompt()
        self.invocation_count = 0

    def _create_prompt(self) -> ChatPromptTemplate:
        return ChatPromptTemplate.from_messages([
            ("system", """You are a regulatory compliance expert for the insurance industry.
You specialize in analyzing IRDAI (India) and MAS (Singapore) regulations.

Your task is to analyze regulatory text and extract:
1. A clear summary of what the regulation requires
2. The compliance domain (governance, risk_management, information_security, audit, fraud, reinsurance, claims, investments, sales)
3. Specific mandatory actions for compliance
4. Evidence/documentation needed to demonstrate compliance
5. Risk level if not compliant (LOW, MEDIUM, HIGH, CRITICAL)
6. Timeline for compliance (if specified)

{format_instructions}"""),
            ("human", """Analyze this regulatory text and provide a structured analysis:

REGULATORY TEXT:
{regulation_text}

Provide your analysis following the specified format.""")
        ])

    def analyze(self, regulation_text: str, metadata: Dict = None) -> RegulationAnalysis:
        """
        Analyze a regulatory requirement with full tracing.

        Args:
            regulation_text: The regulatory text to analyze
            metadata: Optional metadata to include in trace

        Returns:
            RegulationAnalysis with structured output
        """
        self.invocation_count += 1

        # Get Langfuse callback handler
        handler = get_langfuse_handler(
            trace_name=f"regulation-monitor-{self.invocation_count}",
            tags=["agent", "regulation", "analysis"],
            metadata=metadata
        )

        # Build LCEL chain
        chain = self.prompt | self.llm | self.parser

        # Invoke with tracing
        result = chain.invoke(
            {
                "regulation_text": regulation_text[:6000],
                "format_instructions": self.parser.get_format_instructions()
            },
            config={"callbacks": [handler]}
        )

        # Log confidence score to Langfuse
        if handler.trace:
            langfuse.score(
                name="regulation-analysis-confidence",
                value=result.confidence,
                trace_id=handler.trace.id
            )

        return result

# Initialize agent
regulation_agent = RegulationMonitorAgent()
print("Regulation Monitor Agent initialized")

## 4.3 Gap Detector Agent

In [ ]:
# ============================================================================
# WEEK 4.3: GAP DETECTOR AGENT
# ============================================================================

class GapDetectorAgent:
    """
    Agent for detecting compliance gaps.

    Responsibilities:
    - Compare regulatory requirements against company policies
    - Identify gaps where policies don't meet requirements
    - Assess gap severity
    - Provide remediation recommendations

    All invocations are traced to Langfuse.
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model_name = model
        self.llm = ChatOpenAI(model=model, temperature=0)
        self.parser = PydanticOutputParser(pydantic_object=GapAssessment)
        self.prompt = self._create_prompt()
        self.invocation_count = 0

    def _create_prompt(self) -> ChatPromptTemplate:
        return ChatPromptTemplate.from_messages([
            ("system", """You are a compliance gap analyst specializing in insurance regulation.

Your task is to compare regulatory requirements against company policies to identify gaps.

For each comparison:
1. Understand what the regulation requires
2. Assess if the policy addresses the requirement
3. Identify any gaps (missing coverage, insufficient detail, etc.)
4. Assess gap severity based on regulatory risk
5. Provide specific remediation steps
6. Estimate effort to remediate

Severity Guidelines:
- CRITICAL: Could result in license revocation or major fines
- HIGH: Could result in regulatory action or significant fines
- MEDIUM: Could result in findings in audit
- LOW: Minor gap, best practice recommendation

{format_instructions}"""),
            ("human", """Compare this regulatory requirement against the company policy:

REGULATORY REQUIREMENT:
{regulation}

COMPANY POLICY:
{policy}

Identify any compliance gaps and provide your assessment.""")
        ])

    def assess(self, regulation: str, policy: str, metadata: Dict = None) -> GapAssessment:
        """
        Assess compliance gap with full tracing.

        Args:
            regulation: Regulatory requirement text
            policy: Company policy text
            metadata: Optional metadata for trace

        Returns:
            GapAssessment with structured output
        """
        self.invocation_count += 1

        handler = get_langfuse_handler(
            trace_name=f"gap-detector-{self.invocation_count}",
            tags=["agent", "gap", "assessment"],
            metadata=metadata
        )

        chain = self.prompt | self.llm | self.parser

        result = chain.invoke(
            {
                "regulation": regulation[:4000],
                "policy": policy[:4000],
                "format_instructions": self.parser.get_format_instructions()
            },
            config={"callbacks": [handler]}
        )

        # Log severity score
        severity_scores = {"LOW": 0.25, "MEDIUM": 0.5, "HIGH": 0.75, "CRITICAL": 1.0}
        if handler.trace:
            langfuse.score(
                name="gap-severity",
                value=severity_scores.get(result.severity, 0.5),
                trace_id=handler.trace.id
            )

        return result

# Initialize agent
gap_agent = GapDetectorAgent()
print("Gap Detector Agent initialized")

## 4.4 Policy Analyzer Agent

In [ ]:
# ============================================================================
# WEEK 4.4: POLICY ANALYZER AGENT
# ============================================================================

class PolicyAnalyzerAgent:
    """
    Agent for analyzing company policies for compliance coverage.

    Responsibilities:
    - Evaluate policies against regulatory requirements
    - Calculate compliance scores by domain
    - Identify critical gaps
    - Provide improvement recommendations

    All invocations are traced to Langfuse.
    """

    def __init__(self, model: str = "gpt-4o-mini"):
        self.model_name = model
        self.llm = ChatOpenAI(model=model, temperature=0)
        self.parser = PydanticOutputParser(pydantic_object=ComplianceScore)
        self.prompt = self._create_prompt()
        self.invocation_count = 0

    def _create_prompt(self) -> ChatPromptTemplate:
        return ChatPromptTemplate.from_messages([
            ("system", """You are a policy compliance analyst for insurance companies.

Your task is to evaluate company policies against regulatory requirements and provide a compliance score.

Scoring Guidelines:
- 90-100: COMPLIANT - Fully addresses all requirements
- 70-89: PARTIALLY_COMPLIANT - Addresses most requirements with minor gaps
- 50-69: PARTIALLY_COMPLIANT - Significant gaps exist
- 0-49: NON_COMPLIANT - Major gaps, does not meet requirements

{format_instructions}"""),
            ("human", """Evaluate this policy against the regulations for compliance:

COMPLIANCE DOMAIN: {domain}

POLICY CONTENT:
{policy}

RELEVANT REGULATIONS:
{regulations}

Provide a compliance score and detailed assessment.""")
        ])

    def analyze(self, domain: str, policy: str, regulations: str,
                metadata: Dict = None) -> ComplianceScore:
        """
        Analyze policy compliance with full tracing.

        Args:
            domain: Compliance domain being assessed
            policy: Company policy text
            regulations: Relevant regulatory text
            metadata: Optional metadata for trace

        Returns:
            ComplianceScore with structured output
        """
        self.invocation_count += 1

        handler = get_langfuse_handler(
            trace_name=f"policy-analyzer-{self.invocation_count}",
            tags=["agent", "policy", domain],
            metadata=metadata
        )

        chain = self.prompt | self.llm | self.parser

        result = chain.invoke(
            {
                "domain": domain,
                "policy": policy[:4000],
                "regulations": regulations[:4000],
                "format_instructions": self.parser.get_format_instructions()
            },
            config={"callbacks": [handler]}
        )

        # Log compliance score
        if handler.trace:
            langfuse.score(
                name="compliance-score",
                value=result.score / 100.0,
                trace_id=handler.trace.id
            )

        return result

# Initialize agent
policy_agent = PolicyAnalyzerAgent()
print("Policy Analyzer Agent initialized")
print("\nAll three compliance agents ready!")

## 4.5 Agent Invocation Examples

In [ ]:
# ============================================================================
# EXAMPLE 1: REGULATION MONITOR AGENT
# ============================================================================

print("EXAMPLE 1: Regulation Monitor Agent")
print("=" * 60)

test_regulations = [
    """The Board of Directors shall establish a Risk Management Committee comprising
    at least three independent directors. The committee shall meet quarterly to review
    the company's risk exposure and mitigation strategies. Minutes of meetings must be
    maintained for a period of not less than 8 years.""",

    """Insurers must implement comprehensive cybersecurity controls including but not
    limited to: multi-factor authentication for all systems accessing customer data,
    encryption of data at rest and in transit using AES-256 or equivalent, regular
    penetration testing at least annually, and documented incident response procedures.""",

    """The company shall maintain capital reserves as specified in the solvency margin
    requirements. The required solvency margin shall be 150% of the minimum capital
    requirement. Quarterly reports on capital adequacy must be submitted to the
    regulatory authority within 45 days of quarter end."""
]

for i, reg_text in enumerate(test_regulations, 1):
    print(f"\n[Invocation {i}]")
    print(f"Regulation: {reg_text[:80]}...")

    result = regulation_agent.analyze(reg_text, metadata={"example": i})

    print(f"\n  Summary: {result.requirement_summary[:100]}...")
    print(f"  Domain: {result.compliance_domain}")
    print(f"  Risk Level: {result.risk_level}")
    print(f"  Timeline: {result.timeline}")
    print(f"  Mandatory Actions: {len(result.mandatory_actions)}")
    for action in result.mandatory_actions[:2]:
        print(f"    - {action[:60]}...")
    print(f"  Confidence: {result.confidence:.2f}")

print(f"\nTotal Regulation Agent invocations: {regulation_agent.invocation_count}")
langfuse.flush()

In [ ]:
# ============================================================================
# EXAMPLE 2: GAP DETECTOR AGENT
# ============================================================================

print("EXAMPLE 2: Gap Detector Agent")
print("=" * 60)

gap_scenarios = [
    {
        "regulation": """The Board shall have an independent audit committee meeting monthly
        with documented minutes. The committee must comprise at least 3 independent directors
        with at least one having financial expertise.""",
        "policy": """The company has a finance committee with internal members that meets
        quarterly to review financial statements. Meeting notes are kept informally."""
    },
    {
        "regulation": """All customer data must be encrypted at rest and in transit using
        AES-256 encryption. Encryption keys must be rotated every 90 days.""",
        "policy": """Customer data is protected using standard security measures.
        The IT team implements appropriate encryption where feasible."""
    },
    {
        "regulation": """Claims must be settled within 30 days of receipt of all required
        documents. Any delay must be communicated to the policyholder with reasons.""",
        "policy": """Claims are processed in a timely manner based on complexity.
        The claims team aims to resolve claims as quickly as possible."""
    }
]

for i, scenario in enumerate(gap_scenarios, 1):
    print(f"\n[Invocation {i}]")
    print(f"Regulation: {scenario['regulation'][:60]}...")
    print(f"Policy: {scenario['policy'][:60]}...")

    result = gap_agent.assess(
        scenario['regulation'],
        scenario['policy'],
        metadata={"example": i}
    )

    print(f"\n  Gap Identified: {result.gap_identified}")
    print(f"  Severity: {result.severity}")
    print(f"  Gap Description: {result.gap_description[:100]}...")
    print(f"  Effort to Remediate: {result.estimated_effort}")
    print(f"  Remediation Steps: {len(result.remediation_steps)}")
    print(f"  Confidence: {result.confidence:.2f}")

print(f"\nTotal Gap Detector invocations: {gap_agent.invocation_count}")
langfuse.flush()

In [ ]:
# ============================================================================
# EXAMPLE 3: POLICY ANALYZER AGENT
# ============================================================================

print("EXAMPLE 3: Policy Analyzer Agent")
print("=" * 60)

# Sample policy and regulations for governance domain
sample_policy = """
GOVERNANCE POLICY

1. Board Structure
The Board of Directors comprises 9 members, including 4 independent directors.
The Chairman is an independent director.

2. Board Committees
- Audit Committee: 3 members, all independent
- Risk Committee: 4 members, 2 independent
- Nomination Committee: 3 members

3. Meeting Frequency
The Board meets quarterly. Committees meet as needed.

4. Documentation
Minutes are maintained for all Board meetings.
"""

sample_regulations = """
GOVERNANCE REQUIREMENTS:

1. The Board shall have at least 50% independent directors.
2. The Chairman should preferably be an independent director.
3. Risk Management Committee must have at least 3 independent directors.
4. All committees must meet at least quarterly.
5. Minutes must be maintained for 8 years minimum.
"""

result = policy_agent.analyze(
    domain="governance",
    policy=sample_policy,
    regulations=sample_regulations,
    metadata={"example": "governance-assessment"}
)

print(f"\nDomain: {result.domain}")
print(f"Score: {result.score}/100")
print(f"Status: {result.status}")
print(f"Gaps Count: {result.gaps_count}")
print(f"\nCritical Gaps:")
for gap in result.critical_gaps:
    print(f"  - {gap}")
print(f"\nRecommendations:")
for rec in result.recommendations[:3]:
    print(f"  - {rec}")

print(f"\nTotal Policy Analyzer invocations: {policy_agent.invocation_count}")
langfuse.flush()

## Checkpoint: Week 4 Verification

In [ ]:
# ============================================================================
# WEEK 4 CHECKPOINT
# ============================================================================

print("WEEK 4 CHECKPOINT - VERIFICATION")
print("=" * 60)

checkpoints = {
    "Pydantic models defined": all(m in globals() for m in ['RegulationAnalysis', 'GapAssessment', 'ComplianceScore']),
    "Regulation Monitor Agent": 'regulation_agent' in globals() and regulation_agent.invocation_count > 0,
    "Gap Detector Agent": 'gap_agent' in globals() and gap_agent.invocation_count > 0,
    "Policy Analyzer Agent": 'policy_agent' in globals() and policy_agent.invocation_count > 0,
    "Langfuse tracing active": True
}

all_passed = True
for check, passed in checkpoints.items():
    status = "PASS" if passed else "FAIL"
    symbol = "[x]" if passed else "[ ]"
    print(f"  {symbol} {check}: {status}")
    if not passed:
        all_passed = False

print("\n" + "=" * 60)
print(f"Agent Invocation Summary:")
print(f"  Regulation Monitor: {regulation_agent.invocation_count}")
print(f"  Gap Detector: {gap_agent.invocation_count}")
print(f"  Policy Analyzer: {policy_agent.invocation_count}")

if all_passed:
    print("\nWeek 4 COMPLETE - Ready to proceed to Week 5!")
else:
    print("\nWARNING: Some checkpoints failed.")

checkpoint_trace = langfuse.trace(name="week4-checkpoint", session_id=SESSION_ID, tags=["checkpoint", "week4"])
checkpoint_trace.update(output={"checkpoints": checkpoints, "all_passed": all_passed})
langfuse.flush()

## Week 4 Summary

### What We Accomplished

| Component | Status | Key Output |
|-----------|--------|------------|
| Pydantic Models | Complete | 3 structured output models |
| Regulation Monitor | Complete | Analyzes regulatory requirements |
| Gap Detector | Complete | Identifies compliance gaps |
| Policy Analyzer | Complete | Scores policy compliance |

### Key Learnings

1. **Structured Outputs**: Pydantic ensures consistent, validated agent responses
2. **LCEL Chains**: LangChain Expression Language provides clean, traceable chains
3. **Scoring Integration**: Langfuse scores enable compliance metrics tracking

### Next Week Preview

In Week 5, we will:
- Build a multi-agent orchestrator
- Implement domain-level compliance assessment workflows
- Add retry logic and error handling

---
# WEEK 5: Multi-Agent Orchestration (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 5: Multi-Agent Orchestration

---

## Learning Objectives

By the end of Week 5, you will be able to:

- [ ] Build a multi-agent orchestrator that coordinates compliance agents
- [ ] Implement domain-level compliance assessment workflows
- [ ] Add retry logic with exponential backoff
- [ ] Run full compliance assessments across multiple domains

In [ ]:
# ============================================================================
# WEEK 5: COMPLIANCE ORCHESTRATOR
# ============================================================================

from tenacity import retry, stop_after_attempt, wait_exponential

class ComplianceOrchestrator:
    """
    Multi-agent orchestrator for compliance assessment.
    Coordinates Regulation Monitor, Gap Detector, and Policy Analyzer agents.
    All operations are fully traced to Langfuse.
    """

    def __init__(self):
        self.regulation_agent = regulation_agent
        self.gap_agent = gap_agent
        self.policy_agent = policy_agent
        self.vector_store = compliance_vector_store
        self.orchestration_count = 0

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def _analyze_regulation_with_retry(self, text: str) -> Dict:
        """Analyze regulation with retry logic."""
        try:
            result = self.regulation_agent.analyze(text)
            return {'success': True, 'result': result.model_dump()}
        except Exception as e:
            return {'success': False, 'error': str(e)}

    @retry(stop=stop_after_attempt(3), wait=wait_exponential(multiplier=1, min=2, max=10))
    def _detect_gap_with_retry(self, regulation: str, policy: str) -> Dict:
        """Detect gap with retry logic."""
        try:
            result = self.gap_agent.assess(regulation, policy)
            return {'success': True, 'result': result.model_dump()}
        except Exception as e:
            return {'success': False, 'error': str(e)}

    def assess_domain_compliance(self, domain: str) -> Dict:
        """
        Assess compliance for a specific domain.

        Workflow:
        1. Find relevant regulations for the domain
        2. Find matching policies
        3. Analyze regulations
        4. Detect gaps between regulations and policies
        5. Calculate overall score
        """
        self.orchestration_count += 1

        trace = langfuse.trace(
            name=f"orchestration-{domain}-{self.orchestration_count}",
            session_id=SESSION_ID,
            tags=["orchestration", "domain-assessment", domain],
            input={"domain": domain}
        )

        results = {
            'domain': domain,
            'timestamp': datetime.now().isoformat(),
            'regulations_analyzed': 0,
            'gaps_found': [],
            'overall_score': None
        }

        # Step 1: Find relevant regulations
        span_search = trace.span(name="search-regulations")
        reg_results = self.vector_store.find_relevant_regulations(
            f"{domain} compliance requirements insurance", 3
        )
        span_search.end(output={"results": len(reg_results.get('documents', [[]])[0])})

        # Step 2: Find matching policies
        pol_results = self.vector_store.find_matching_policies(
            f"{domain} policy procedures", 2
        )

        # Step 3 & 4: Analyze regulations and detect gaps
        if reg_results['documents'] and reg_results['documents'][0]:
            results['regulations_analyzed'] = len(reg_results['documents'][0])

            for reg_text in reg_results['documents'][0][:2]:
                # Analyze regulation
                analysis = self._analyze_regulation_with_retry(reg_text)

                # Detect gaps against policies
                if pol_results['documents'] and pol_results['documents'][0]:
                    for pol_text in pol_results['documents'][0][:1]:
                        gap_result = self._detect_gap_with_retry(reg_text, pol_text)
                        if gap_result.get('success') and gap_result['result'].get('gap_identified'):
                            results['gaps_found'].append(gap_result['result'])

        # Step 5: Calculate score
        total_gaps = len(results['gaps_found'])
        critical_gaps = sum(1 for g in results['gaps_found'] if g.get('severity') in ['HIGH', 'CRITICAL'])

        if results['regulations_analyzed'] > 0:
            base_score = 100 - (total_gaps * 15) - (critical_gaps * 20)
            results['overall_score'] = max(0, min(100, base_score))

        trace.update(output={
            "domain": domain,
            "score": results['overall_score'],
            "gaps_count": total_gaps,
            "critical_gaps": critical_gaps
        })

        if results['overall_score'] is not None:
            langfuse.score(
                name="domain-compliance-score",
                value=results['overall_score'] / 100.0,
                trace_id=trace.id
            )

        return results

# Initialize orchestrator
compliance_orchestrator = ComplianceOrchestrator()
print("Compliance Orchestrator initialized")

In [ ]:
# ============================================================================
# ORCHESTRATION EXAMPLE 1: Single Domain Assessment
# ============================================================================

print("ORCHESTRATION EXAMPLE 1: Single Domain Assessment")
print("=" * 60)

domain_result = compliance_orchestrator.assess_domain_compliance('governance')

print(f"\nDomain: {domain_result['domain']}")
print(f"Overall Score: {domain_result['overall_score']}/100")
print(f"Regulations Analyzed: {domain_result['regulations_analyzed']}")
print(f"Gaps Found: {len(domain_result['gaps_found'])}")

if domain_result['gaps_found']:
    print("\nGap Details:")
    for i, gap in enumerate(domain_result['gaps_found'][:3], 1):
        print(f"  {i}. Severity: {gap.get('severity')}")
        print(f"     {gap.get('gap_description', '')[:100]}...")

langfuse.flush()

In [ ]:
# ============================================================================
# ORCHESTRATION EXAMPLE 2: Multi-Domain Assessment
# ============================================================================

print("ORCHESTRATION EXAMPLE 2: Multi-Domain Assessment")
print("=" * 60)

domains_to_assess = ['risk_management', 'information_security', 'audit']

domain_scores = {}
for domain in domains_to_assess:
    print(f"\nAssessing: {domain}...")
    result = compliance_orchestrator.assess_domain_compliance(domain)
    domain_scores[domain] = result
    print(f"  Score: {result['overall_score']}/100")
    print(f"  Gaps: {len(result['gaps_found'])}")

print("\n" + "=" * 60)
print("MULTI-DOMAIN COMPLIANCE SUMMARY")
print("=" * 60)
for domain, result in domain_scores.items():
    score = result['overall_score'] or 0
    status = "COMPLIANT" if score >= 80 else "NEEDS ATTENTION" if score >= 60 else "AT RISK"
    bar = '=' * int(score / 5)
    print(f"  {domain:20} {score:3}/100 [{status:15}] {bar}")

langfuse.flush()

## Checkpoint: Week 5 Verification

In [ ]:
# ============================================================================
# WEEK 5 CHECKPOINT
# ============================================================================

print("WEEK 5 CHECKPOINT")
print("=" * 60)

checkpoints = {
    "Orchestrator initialized": 'compliance_orchestrator' in globals(),
    "Single domain assessment": compliance_orchestrator.orchestration_count >= 1,
    "Multi-domain assessment": len(domain_scores) >= 3,
    "Retry logic implemented": True
}

all_passed = all(checkpoints.values())
for check, passed in checkpoints.items():
    symbol = "[x]" if passed else "[ ]"
    print(f"  {symbol} {check}")

print(f"\nOrchestration invocations: {compliance_orchestrator.orchestration_count}")
if all_passed:
    print("\nWeek 5 COMPLETE!")

langfuse.flush()

---
# WEEK 6: Compliance Knowledge Graph (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 6: Compliance Knowledge Graph

---

## Learning Objectives

By the end of Week 6, you will be able to:

- [ ] Build a knowledge graph of regulation-policy relationships
- [ ] Visualize compliance structure using NetworkX and PyVis
- [ ] Query the graph for compliance insights

In [ ]:
# ============================================================================
# WEEK 6: COMPLIANCE KNOWLEDGE GRAPH
# ============================================================================

import networkx as nx
from pyvis.network import Network

class ComplianceKnowledgeGraph:
    """
    Knowledge graph for regulatory compliance relationships.
    Maps regulators -> domains -> regulations -> policies.
    """

    def __init__(self):
        trace = langfuse.trace(
            name="init-knowledge-graph",
            session_id=SESSION_ID,
            tags=["knowledge-graph", "init"]
        )

        self.graph = nx.DiGraph()
        self._build_compliance_ontology()

        trace.update(output={"nodes": self.graph.number_of_nodes()})

    def _build_compliance_ontology(self):
        """Build base compliance ontology."""
        # Add regulators
        for reg in ['IRDAI', 'MAS']:
            self.graph.add_node(reg, type='REGULATOR', color='#e74c3c')

        # Add domains
        for domain in COMPLIANCE_DOMAINS.keys():
            self.graph.add_node(domain, type='DOMAIN', color='#3498db')

        # Connect regulators to domains
        irdai_domains = ['governance', 'reinsurance', 'claims', 'audit', 'fraud', 'investments']
        mas_domains = ['governance', 'risk_management', 'information_security', 'sales']

        for domain in irdai_domains:
            if domain in COMPLIANCE_DOMAINS:
                self.graph.add_edge('IRDAI', domain, relation='REGULATES')

        for domain in mas_domains:
            if domain in COMPLIANCE_DOMAINS:
                self.graph.add_edge('MAS', domain, relation='REGULATES')

    def add_regulation(self, doc: Dict):
        """Add a regulation document to the graph."""
        doc_id = doc.get('doc_id')
        self.graph.add_node(
            doc_id,
            type='REGULATION',
            filename=doc.get('filename'),
            color='#2ecc71'
        )

        # Link to domains based on filename
        filename_lower = doc.get('filename', '').lower()
        domain_keywords = {
            'governance': ['governance', 'corporate', 'board'],
            'risk_management': ['risk'],
            'information_security': ['security', 'cyber', 'information'],
            'audit': ['audit', 'control'],
            'fraud': ['fraud'],
            'reinsurance': ['reinsurance'],
            'claims': ['claim', 'grievance'],
            'investments': ['investment', 'financial']
        }

        for domain, keywords in domain_keywords.items():
            if any(kw in filename_lower for kw in keywords):
                self.graph.add_edge(doc_id, domain, relation='COVERS')

    def add_policy(self, doc: Dict):
        """Add a policy document to the graph."""
        doc_id = doc.get('doc_id')
        self.graph.add_node(
            doc_id,
            type='POLICY',
            filename=doc.get('filename'),
            color='#f39c12'
        )

    def get_stats(self) -> Dict:
        """Get graph statistics."""
        return {
            'total_nodes': self.graph.number_of_nodes(),
            'total_edges': self.graph.number_of_edges(),
            'regulators': len([n for n, d in self.graph.nodes(data=True) if d.get('type') == 'REGULATOR']),
            'domains': len([n for n, d in self.graph.nodes(data=True) if d.get('type') == 'DOMAIN']),
            'regulations': len([n for n, d in self.graph.nodes(data=True) if d.get('type') == 'REGULATION']),
            'policies': len([n for n, d in self.graph.nodes(data=True) if d.get('type') == 'POLICY'])
        }

# Build knowledge graph
compliance_kg = ComplianceKnowledgeGraph()

for doc in all_compliance_docs['regulations']:
    compliance_kg.add_regulation(doc)

for doc in all_compliance_docs['policies']:
    compliance_kg.add_policy(doc)

stats = compliance_kg.get_stats()
print("COMPLIANCE KNOWLEDGE GRAPH")
print("=" * 50)
for key, value in stats.items():
    print(f"  {key}: {value}")

langfuse.flush()

In [ ]:
# ============================================================================
# KNOWLEDGE GRAPH VISUALIZATION (MATPLOTLIB)
# ============================================================================

print("KNOWLEDGE GRAPH VISUALIZATION")
print("=" * 50)

fig, ax = plt.subplots(1, 1, figsize=(16, 12))

color_map = {
    'REGULATOR': '#e74c3c',
    'DOMAIN': '#3498db',
    'REGULATION': '#2ecc71',
    'POLICY': '#f39c12'
}

node_colors = []
for node in compliance_kg.graph.nodes():
    node_type = compliance_kg.graph.nodes[node].get('type', 'OTHER')
    node_colors.append(color_map.get(node_type, '#95a5a6'))

pos = nx.spring_layout(compliance_kg.graph, k=2, iterations=50, seed=42)

nx.draw_networkx_nodes(compliance_kg.graph, pos, node_color=node_colors, node_size=1500, alpha=0.9, ax=ax)
nx.draw_networkx_edges(compliance_kg.graph, pos, edge_color='#bdc3c7', arrows=True, arrowsize=15, alpha=0.6, ax=ax)

labels = {n: n[:12] for n in compliance_kg.graph.nodes()}
nx.draw_networkx_labels(compliance_kg.graph, pos, labels, font_size=8, ax=ax)

legend_elements = [
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c', markersize=15, label='Regulator'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#3498db', markersize=15, label='Domain'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71', markersize=15, label='Regulation'),
    plt.Line2D([0], [0], marker='o', color='w', markerfacecolor='#f39c12', markersize=15, label='Policy')
]
ax.legend(handles=legend_elements, loc='upper left')
ax.set_title('Compliance Knowledge Graph', fontsize=14, fontweight='bold')
ax.axis('off')

plt.tight_layout()
plt.savefig('compliance_knowledge_graph.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nGraph saved to compliance_knowledge_graph.png")

In [ ]:
# ============================================================================
# INTERACTIVE PYVIS VISUALIZATION
# ============================================================================

print("Creating interactive visualization...")

net = Network(height="600px", width="100%", directed=True, notebook=True)

for node, data in compliance_kg.graph.nodes(data=True):
    node_type = data.get('type', 'OTHER')
    color = color_map.get(node_type, '#95a5a6')
    label = str(node)[:15]
    title = f"{node_type}: {data.get('filename', node)}"
    net.add_node(node, label=label, color=color, title=title, size=25)

for source, target, data in compliance_kg.graph.edges(data=True):
    relation = data.get('relation', '')
    net.add_edge(source, target, title=relation)

net.set_options('''
var options = {
  "physics": {
    "forceAtlas2Based": {"gravitationalConstant": -50, "centralGravity": 0.01},
    "solver": "forceAtlas2Based"
  }
}
''')

net.save_graph("compliance_knowledge_graph_interactive.html")
print("Interactive graph saved to compliance_knowledge_graph_interactive.html")

langfuse.flush()

## Week 6 Checkpoint

In [ ]:
print("WEEK 6 CHECKPOINT")
print("=" * 60)

checkpoints = {
    "Knowledge graph initialized": 'compliance_kg' in globals(),
    "Regulations added": compliance_kg.get_stats()['regulations'] > 0,
    "Static visualization": True,
    "Interactive visualization": True
}

for check, passed in checkpoints.items():
    symbol = "[x]" if passed else "[ ]"
    print(f"  {symbol} {check}")

if all(checkpoints.values()):
    print("\nWeek 6 COMPLETE!")

---
# WEEK 7: Advanced Observability (PROVIDED)
---

*Run these cells to set up the foundation for Week 8.*

---

# WEEK 7: Advanced Observability

---

## Learning Objectives

- [ ] Create a comprehensive observability dashboard
- [ ] Track all metrics in Langfuse
- [ ] Generate compliance audit reports

In [ ]:
# ============================================================================
# WEEK 7: OBSERVABILITY DASHBOARD
# ============================================================================

print("COMPLIANCE OBSERVABILITY DASHBOARD")
print("=" * 60)

observability_metrics = {
    'session_id': SESSION_ID,
    'timestamp': datetime.now().isoformat(),
    'documents': {
        'regulations_processed': len(all_compliance_docs['regulations']),
        'policies_processed': len(all_compliance_docs['policies']),
        'requirements_extracted': len(all_requirements) if 'all_requirements' in globals() else 0
    },
    'agents': {
        'regulation_monitor_invocations': regulation_agent.invocation_count,
        'gap_detector_invocations': gap_agent.invocation_count,
        'policy_analyzer_invocations': policy_agent.invocation_count
    },
    'orchestration': {
        'total_invocations': compliance_orchestrator.orchestration_count,
        'domains_assessed': len(domain_scores) if 'domain_scores' in globals() else 0
    },
    'knowledge_graph': compliance_kg.get_stats()
}

# Log to Langfuse
metrics_trace = langfuse.trace(
    name="observability-dashboard",
    session_id=SESSION_ID,
    tags=["observability", "metrics", "dashboard"]
)
metrics_trace.update(output=observability_metrics)

print(f"\nSession: {SESSION_ID}")
print(f"\nDocuments:")
for key, val in observability_metrics['documents'].items():
    print(f"  {key}: {val}")

print(f"\nAgent Invocations:")
for key, val in observability_metrics['agents'].items():
    print(f"  {key}: {val}")

print(f"\nOrchestration:")
for key, val in observability_metrics['orchestration'].items():
    print(f"  {key}: {val}")

print(f"\nKnowledge Graph:")
for key, val in observability_metrics['knowledge_graph'].items():
    print(f"  {key}: {val}")

langfuse.flush()

---

# WEEK 8: Production Deployment

---

## Learning Objectives

- [ ] Build a Gradio UI for the compliance system
- [ ] Deploy the complete system
- [ ] Verify end-to-end functionality

---
# WEEK 8: Production Deployment (YOUR WORK)
---

## Instructions

Complete the TODO cells below. Each cell has hints and instructions.
Refer to the learning objectives above and the master notebook for guidance.

In [ ]:
# ============================================================================
# TODO 8.1: WEEK 8: INTEGRATED COMPLIANCE SYSTEM
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
# HINT: class RegulatoryComplianceSystem:
# HINT: def __init__(self):
# HINT: def assess_domain(self, domain: str) -> Dict:
# HINT: def search_regulations(self, query: str) -> Dict:
# HINT: def search_policies(self, query: str) -> Dict:
# HINT: def get_status(self) -> Dict:
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement WEEK 8: INTEGRATED COMPLIANCE SYSTEM
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.2: GRADIO UI
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
# HINT: def assess_domain_ui(domain: str) -> str:
# HINT: def search_regulations_ui(query: str) -> str:
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement GRADIO UI
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.3: LAUNCH GRADIO (Optional)
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement LAUNCH GRADIO (Optional)
raise NotImplementedError('Complete this section')

## Final Checkpoint: Week 8 & Complete System

---## Production Deployment: FastAPI ApplicationNow that we have a working Gradio prototype, let's create a **production-ready deployment** using:- **FastAPI** — Industry-standard async API framework- **Animated Web UI** — Beautiful interface showing the RAG pipeline flow in real-time- **Docker** — Containerized deployment ready for AWS/GCPThe web interface features animated visualizations of:- Document processing and chunking- Vector embedding and semantic search- Multi-agent orchestration and routing- Response generation pipeline---

In [ ]:
# ============================================================================
# TODO 8.4: PRODUCTION DEPLOYMENT: FastAPI Application
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement PRODUCTION DEPLOYMENT: FastAPI Application
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.5: ---------------------------------------------------------------------------
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
# HINT: def _lf_trace(name: str, **kwargs):
# HINT: def _lf_generation(trace, name: str, model: str, input_data):
# HINT: def _effective_price(p: dict) -> float:
# HINT: def _product_text(p: dict) -> str:
# HINT: def _init_vector_store():
# HINT: def _batch_embed(texts: List[str]) -> List[List[float]]:
# HINT: def _embed(text: str) -> List[float]:
# HINT: def _search(query: str, n: int = 5, where: dict = None) -> List[dict]:
# HINT: def _traced_completion(messages: List[dict], trace_name: str, model: str = "gpt-4o-mini") -> str:
# HINT: def _run_price_agent(category: str, context: str) -> dict:
# HINT: def _run_catalog_agent(category: str, context: str) -> dict:
# HINT: def _run_marketing_agent(query: str, context: str) -> dict:
# HINT: def run_pipeline(query: str) -> dict:
# HINT: class QueryRequest(BaseModel):
# HINT: class QueryResponse(BaseModel):
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement ---------------------------------------------------------------------------
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.6: Week 8 Task 6
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 6
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.7: Week 8 Task 7
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 7
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.8: Week 8 Task 8
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 8
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# TODO 8.9: Week 8 Task 9
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement Week 8 Task 9
raise NotImplementedError('Complete this section')

---

## Production Deployment: AWS EC2 Guide

After running the FastAPI cells above, you have a `deployment/` folder with everything needed. Here's how to deploy it to AWS EC2.

### Step 1: Launch an EC2 Instance

1. Go to **AWS Console → EC2 → Launch Instance**
2. **AMI**: Ubuntu Server 22.04 LTS (Free Tier eligible)
3. **Instance type**: `t2.micro` (Free Tier) for testing, `t3.small` or `t3.medium` for production
4. **Key pair**: Create or select an existing `.pem` key pair — download it
5. **Network settings**:
   - Allow SSH (port 22) from your IP
   - Add a **Custom TCP rule** for **port 8000** from `0.0.0.0/0` (or your IP for security)
6. **Storage**: 20 GB gp3 (default is fine)
7. Click **Launch Instance**

### Step 2: Transfer Files to EC2

**Option A: From your local machine (after downloading deployment/ from Colab)**

```bash
# Make your key file secure
chmod 400 your-key.pem

# Copy the deployment folder to EC2
scp -i your-key.pem -r deployment/ ubuntu@<EC2-PUBLIC-IP>:~/app/
```

**Option B: From Google Colab**

```python
# In Colab, zip the deployment folder
!zip -r deployment.zip deployment/

# Download it to your local machine
from google.colab import files
files.download('deployment.zip')

# Then scp from local to EC2 as shown in Option A
```

### Step 3: SSH into EC2 and Set Up

```bash
# Connect to your instance
ssh -i your-key.pem ubuntu@<EC2-PUBLIC-IP>

# Update system and install Python
sudo apt update && sudo apt upgrade -y
sudo apt install -y python3-pip python3-venv

# Set up the app
cd ~/app
python3 -m venv venv
source venv/bin/activate
pip install -r requirements.txt
```

### Step 4: Configure Environment Variables

```bash
# Create .env file with your API keys
cat > .env << 'EOF'
OPENAI_API_KEY=sk-your-openai-key
LANGFUSE_SECRET_KEY=sk-lf-your-secret
LANGFUSE_PUBLIC_KEY=pk-lf-your-public
LANGFUSE_HOST=https://cloud.langfuse.com
EOF
```

### Step 5: Launch the Application

```bash
# Test run (foreground)
uvicorn app:app --host 0.0.0.0 --port 8000

# Production run (background, survives SSH disconnect)
nohup uvicorn app:app --host 0.0.0.0 --port 8000 > app.log 2>&1 &

# Or use systemd for auto-restart:
sudo tee /etc/systemd/system/fastapi-app.service << 'EOF'
[Unit]
Description=FastAPI Competitive Intelligence App
After=network.target

[Service]
User=ubuntu
WorkingDirectory=/home/ubuntu/app
Environment="PATH=/home/ubuntu/app/venv/bin"
EnvironmentFile=/home/ubuntu/app/.env
ExecStart=/home/ubuntu/app/venv/bin/uvicorn app:app --host 0.0.0.0 --port 8000
Restart=always

[Install]
WantedBy=multi-user.target
EOF

sudo systemctl enable fastapi-app
sudo systemctl start fastapi-app
sudo systemctl status fastapi-app
```

### Step 6: Access Your Application

Open in your browser:
```
http://<EC2-PUBLIC-IP>:8000
```

API endpoints:
- **Health check**: `http://<EC2-PUBLIC-IP>:8000/api/health`
- **Query API**: `POST http://<EC2-PUBLIC-IP>:8000/api/query`
- **Swagger docs**: `http://<EC2-PUBLIC-IP>:8000/docs`

### Step 7: Verify Deployment

```bash
# From your local machine or any terminal
curl http://<EC2-PUBLIC-IP>:8000/api/health

# Test a query
curl -X POST http://<EC2-PUBLIC-IP>:8000/api/query \
  -H "Content-Type: application/json" \
  -d '{"query": "Analyze competitive pricing"}'
```

### Docker Alternative (Optional)

If you prefer Docker:

```bash
# On EC2
sudo apt install -y docker.io
sudo usermod -aG docker ubuntu
# Log out and back in, then:
cd ~/app
docker build -t intelligence-app .
docker run -d -p 8000:8000 --env-file .env --name app intelligence-app
```

### Security Recommendations for Production

- Use HTTPS with a reverse proxy (nginx + Let's Encrypt)
- Restrict port 8000 to your IP or use a load balancer
- Store API keys in AWS Secrets Manager instead of .env
- Set up CloudWatch monitoring for the EC2 instance
- Use an Elastic IP so the address doesn't change on reboot


In [ ]:
# ============================================================================
# TODO 8.10: DEPLOYMENT INSTRUCTIONS
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement DEPLOYMENT INSTRUCTIONS
raise NotImplementedError('Complete this section')

In [ ]:
# ============================================================================
# FINAL CHECKPOINT
# ============================================================================

print("FINAL CHECKPOINT - COMPLETE SYSTEM VERIFICATION")
print("=" * 60)

final_checkpoints = {
    "Week 1 - Environment & Langfuse": 'langfuse' in globals() and 'SESSION_ID' in globals(),
    "Week 2 - Document Processing": len(all_compliance_docs.get('regulations', [])) > 0,
    "Week 3 - Vector Store": 'compliance_vector_store' in globals(),
    "Week 4 - Compliance Agents": all([regulation_agent.invocation_count > 0, gap_agent.invocation_count > 0]),
    "Week 5 - Orchestration": compliance_orchestrator.orchestration_count > 0,
    "Week 6 - Knowledge Graph": compliance_kg.get_stats()['total_nodes'] > 0,
    "Week 7 - Observability": True,
    "Week 8 - Production System": 'compliance_system' in globals()
}

all_passed = True
for week, passed in final_checkpoints.items():
    status = "PASS" if passed else "FAIL"
    symbol = "[x]" if passed else "[ ]"
    print(f"  {symbol} {week}: {status}")
    if not passed:
        all_passed = False

print("\n" + "=" * 60)
if all_passed:
    print("CONGRATULATIONS! All 8 weeks complete!")
    print("\nYour Regulatory Compliance System is production-ready!")
else:
    print("Some checkpoints need attention.")

# Final trace
final_trace = langfuse.trace(
    name="final-checkpoint",
    session_id=SESSION_ID,
    tags=["checkpoint", "final", "complete"]
)
final_trace.update(output={"checkpoints": final_checkpoints, "all_passed": all_passed})
langfuse.flush()

---

# Summary & Conclusion

---

## What We Built

| Week | Component | Technologies |
|------|-----------|-------------|
| 1 | Environment & Langfuse | API setup, full tracing from start |
| 2 | Document Processing | PDF/DOCX parsing, requirement extraction |
| 3 | Vector Store | ChromaDB, OpenAI embeddings, semantic search |
| 4 | Compliance Agents | Pydantic, LangChain LCEL, 3 specialized agents |
| 5 | Orchestration | Multi-agent workflow, retry logic |
| 6 | Knowledge Graph | NetworkX, PyVis visualization |
| 7 | Observability | Metrics dashboard, audit trails |
| 8 | Production | Gradio UI, integrated system |

## Key Features

- **100% Langfuse Tracing**: Every operation visible in dashboard
- **Structured Outputs**: Pydantic models ensure consistent compliance assessments
- **Semantic Search**: Find related policies/regulations across different terminology
- **Gap Detection**: Automated identification of compliance gaps
- **Compliance Scoring**: Quantified domain-level compliance metrics
- **Knowledge Graph**: Visual map of regulatory relationships

## Next Steps

1. Add more regulations (SOX, GDPR, etc.)
2. Implement continuous monitoring for regulatory changes
3. Build automated remediation workflows
4. Add role-based access control
5. Deploy to cloud infrastructure

In [ ]:
# ============================================================================
# TODO 8.11: FINAL CLEANUP
# ============================================================================
#
# Your task: Implement this section based on the master solution pattern
#
#
# YOUR CODE HERE
# ============================================================================

# TODO: Implement FINAL CLEANUP
raise NotImplementedError('Complete this section')